In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('✅ Set PYTORCH_CUDA_ALLOC_CONF for improved memory management.')
print('   Please restart the Colab runtime (Runtime > Restart session) for this to take effect.')

## Gated Cross-Attention Fusion of CBAM-EfficientNet and Vision Transformer Representations for Explainable Brain Tumor MRI Classificatio

**Dataset:** Br35H (binary: tumor / no tumor)

This is a fixed, Br35H-specific version of your original notebook. What changed and why, briefly:

- **Removed the fragile `timm` cosine scheduler** (its `.step(epoch)` signature has changed across `timm` releases and was a likely crash source) — replaced with a small pure-PyTorch warmup+cosine scheduler.
- **Removed the `torch-ema` dependency** (API drift across versions) — replaced with a ~15-line custom EMA class.
- **Fixed ViT attention rollout** — the original hooked the attention block's *output features*, not the actual attention-probability matrix (which `timm` doesn't expose unless you disable its fused/SDPA attention path). This was silently wrong or could crash on shape mismatches; now fixed properly.
- **Replaced `shap.DeepExplainer` with `shap.GradientExplainer`**, which is far more robust for GELU/SiLU-based hybrid CNN+ViT models.
- **Fixed the LIME predictor** to batch perturbed images in chunks instead of pushing 100+ images through both backbones in one GPU call — this was a likely source of your OOM errors.
- **Added dataset-path validation up front** with a clear error message instead of a deep, confusing crash in the data pipeline.
- **Added `LOW_MEMORY_MODE`** (smaller backbones + smaller batch) and a **`RUN_HEAVY_CELLS`** flag that skips the multi-hour K-Fold CV and ablation study by default — turn it on only once your main pipeline runs cleanly end-to-end.
- Trimmed augmentations that don't make sense for grayscale MRI (hue/saturation jitter, vertical flips).
- Wrapped each individual XAI method in `try/except` so one failing method (e.g. SHAP) doesn't stop you from getting the others.

**New:** Cell 3 now downloads Br35H directly from Kaggle (`ahmedhamada0/brain-tumor-detection`) into the Colab VM — you no longer need to upload the dataset to Drive yourself. You just need a Kaggle API token (see Cell 3 for instructions). `DATASET_PATH` in the Configuration cell is then set automatically.

https://www.kaggle.com/datasets/ahmedhamada0/brain-tumor-detection


## Version 5 changes (final XAI evaluation integrated)

| Where | Change |
|---|---|
| Cell 4b (new) | Writes `br35h_data.py` and `xai_faithfulness_eval.py` to `/content` |
| Cell 5 (Config) | New flags: `SCAN_GROUP_SPLIT = True`, `XAI_MAX_IMAGES = None`, `XAI_RUN_BLACK_SENSITIVITY = False` |
| Cell 6 (data) | `group_id` = scan cluster (exact, rescaled and mirrored copies of a slice share a group). With file-name groups, 397 of 443 test images shared a scan with train. The k-fold CV in Cell 14 uses the same groups |
| Data-preview and Grad-CAM cells | `denorm` detaches before `.numpy()` |
| Cell 15c (new) | Final XAI evaluation: 7 methods, insertion/deletion against uniform and in-brain random maps, class and leakage strata, expert-outline pointing game with chance level, scan-cluster statistics, Table 8 and Figure 8 |
| Removed | Old faithfulness cells (brain-mask pointing game, pooled classes, no random baseline) and the low/high-faithfulness panels; they are superseded by Cell 15c |
| Report cells | Section 4, Section 6 and the final report read the Cell 15c results |

All outputs were cleared: they came from the file-name split and the old XAI code, so none of them describes this notebook. Run top to bottom, including retraining, before taking any number into the manuscript. The original V4 notebook is unchanged.

## Cell 1 — Mount Drive & Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(gpu_info if 'NVIDIA' in gpu_info else '⚠️  No GPU found — go to Runtime > Change runtime type > GPU (T4 is fine)')

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## Cell 2 — Install Libraries

Run once. If this is your **first** install in this runtime, click **Runtime > Restart session** afterward, then continue from Cell 1.

In [ ]:
# Remove the unrelated PyPI package literally named "umap" — it shadows
# umap-learn's "umap" import namespace and is a common source of
# "module umap has no attribute UMAP" errors.
!pip uninstall -y umap -q 2>/dev/null

!pip install -q timm
!pip install -q transformers
!pip install -q grad-cam
!pip install -q shap
!pip install -q lime
!pip install -q captum
!pip install -q albumentations
!pip install -q torchmetrics
!pip install -q scikit-learn imbalanced-learn scikit-image
!pip install -q einops
!pip install -q umap-learn
!pip install -q seaborn plotly
!pip install -q kaggle
!pip install -q imagehash                # pHash — catches resized/recompressed/cropped
                                          # near-duplicates that MD5 exact-hash matching misses
!pip install -q opencv-python-headless   # brain/background segmentation used by the
                                          # shortcut-learning audit and XAI pointing-game check

print("\n✅ All libraries installed!")
print("   If this was a fresh install, restart the runtime now (Runtime > Restart session),")
print("   then re-run from Cell 1 before continuing.")

## Cell 3 — Download the Dataset from Kaggle

Downloads **Br35H :: Brain Tumor Detection 2020** (`ahmedhamada0/brain-tumor-detection` on Kaggle) straight into the Colab VM — no manual upload to Drive needed.

You need a Kaggle API token (kaggle.com → your profile picture → **Settings** → **API** → **Create New Token**, downloads `kaggle.json`). Either:
- Add `KAGGLE_USERNAME` and `KAGGLE_KEY` as **Colab Secrets** (key icon in the left sidebar) and enable them for this notebook — the cell will pick them up automatically, or
- Just run the cell and upload `kaggle.json` when prompted.

In [ ]:
# ── Authenticate with Kaggle ──────────────────────────────────────────────
import os, json, pathlib

os.makedirs('/root/.kaggle', exist_ok=True)
kaggle_json_path = pathlib.Path('/root/.kaggle/kaggle.json')

if not kaggle_json_path.exists():
    creds = None
    try:
        from google.colab import userdata
        creds = {"username": userdata.get('KAGGLE_USERNAME'), "key": userdata.get('KAGGLE_KEY')}
    except Exception:
        creds = None

    if not creds or not creds.get('username') or not creds.get('key'):
        print("No Kaggle credentials found in Colab Secrets.")
        print("Upload your kaggle.json (Kaggle → Settings → API → Create New Token):")
        from google.colab import files
        uploaded = files.upload()
        if 'kaggle.json' not in uploaded:
            raise FileNotFoundError("kaggle.json was not uploaded — re-run this cell and select the file.")
        kaggle_json_path.write_bytes(uploaded['kaggle.json'])
    else:
        kaggle_json_path.write_text(json.dumps(creds))
        print("✅ Kaggle credentials loaded from Colab Secrets.")

os.chmod(kaggle_json_path, 0o600)

# ── Download Br35H :: Brain Tumor Detection 2020 ──────────────────────────
KAGGLE_SLUG  = 'ahmedhamada0/brain-tumor-detection'   # Br35H — folders: yes / no / pred
RAW_DATA_DIR = '/content/br35h_raw'
EXCLUDE_DIRS = {'pred', 'predictions', '.ipynb_checkpoints'}  # not a class — extra unlabeled images

def find_class_root(root, expected={'yes', 'no'}):
    if not os.path.isdir(root):
        return None
    for dirpath, dirnames, _ in os.walk(root):
        names = {d.lower() for d in dirnames}
        if expected.issubset(names):
            return dirpath
    return None

DATASET_PATH = find_class_root(RAW_DATA_DIR)
if DATASET_PATH:
    print(f"✅ Dataset already downloaded at: {DATASET_PATH} (skipping re-download)")
else:
    os.makedirs(RAW_DATA_DIR, exist_ok=True)
    !kaggle datasets download -d {KAGGLE_SLUG} -p {RAW_DATA_DIR} --unzip -q
    DATASET_PATH = find_class_root(RAW_DATA_DIR)
    if DATASET_PATH is None:
        raise FileNotFoundError(
            f"Downloaded the dataset but couldn't find 'yes'/'no' folders under {RAW_DATA_DIR}. "
            f"Check the Colab Files pane for the actual structure and set DATASET_PATH manually "
            f"in the Configuration cell below."
        )
    print(f"✅ Downloaded and extracted to: {DATASET_PATH}")

print("\nClass folders found:")
for d in sorted(os.listdir(DATASET_PATH)):
    full = os.path.join(DATASET_PATH, d)
    if os.path.isdir(full):
        n = len([f for f in os.listdir(full) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        tag = '  (ignored — not a class)' if d.lower() in EXCLUDE_DIRS else ''
        print(f"   {d:<10}: {n} images{tag}")

## Cell 4 — Import All Modules

In [ ]:
import os, sys, random, time, warnings, gc, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageFile
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = True  # don't hard-crash on a slightly corrupted scan

# ── OpenCV — brain/background segmentation for the shortcut-learning audit
#    and the XAI pointing-game check (NOT used anywhere in model training) ──
import cv2

# ── PyTorch ───────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# NOTE: WeightedRandomSampler intentionally removed — see Cell 6. Class
# imbalance is handled via loss-function class weights instead (CombLoss),
# which is more standard on an already near-balanced dataset and avoids the
# sampler silently over/under-sampling images every epoch.

# ── Timm (ViT + EfficientNet) ────────────────────────────────────────────
import timm

# ── Metrics & splitting ──────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GroupShuffleSplit
)
try:
    from sklearn.model_selection import StratifiedGroupKFold
    _HAS_SGKF = True
except ImportError:
    _HAS_SGKF = False  # older sklearn — Cell 14 (K-Fold CV) falls back to GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    confusion_matrix, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score,
    f1_score, precision_score, recall_score
)

# ── XAI ───────────────────────────────────────────────────────────────────
import shap
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM, LayerCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from captum.attr import IntegratedGradients, Occlusion, NoiseTunnel

# ── Near-duplicate detection (perceptual hashing) ─────────────────────────
import imagehash

# ── Augmentation ──────────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Dimensionality Reduction ──────────────────────────────────────────────
from sklearn.manifold import TSNE
from umap import UMAP

# ── AMP: PyTorch moved torch.cuda.amp -> torch.amp; support both ──────────
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    def autocast_ctx(): return _autocast(device_type='cuda', enabled=torch.cuda.is_available())
    def make_scaler():  return _GradScaler('cuda', enabled=torch.cuda.is_available())
except (ImportError, TypeError):
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    def autocast_ctx(): return _autocast(enabled=torch.cuda.is_available())
    def make_scaler():  return _GradScaler(enabled=torch.cuda.is_available())

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def free_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Utility functions for shortcut learning and XAI faithfulness audit ---
def _brain_mask(img_u8):
    gray = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n_cc, labels_cc, stats, _ = cv2.connectedComponentsWithStats(th, connectivity=8)
    if n_cc <= 1:
        return np.ones_like(gray, dtype=np.uint8)
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask = (labels_cc == largest).astype(np.uint8)
    return cv2.dilate(mask, np.ones((5, 5), np.uint8), iterations=2)

def _variant(img_u8, kind):
    mask = _brain_mask(img_u8)
    out = img_u8.copy()
    if kind == 'original':
        return out
    if kind == 'brain_only':
        out[mask == 0] = 0
        return out
    if kind == 'background_only':
        out[mask == 1] = 0
        return out
    if kind == 'skull_masked':
        eroded = cv2.erode(mask, np.ones((9, 9), np.uint8), iterations=2)
        out[eroded == 0] = 0
        return out
    raise ValueError(f'Unknown variant: {kind}')


print(f"✅ All imports successful! Device: {DEVICE}")

## Cell 4b — XAI and data-audit helper modules (added)

The next two cells write `br35h_data.py` (audited Br35H tumor-outline loader, scan-cluster detection, leakage flags, brain mask) and `xai_faithfulness_eval.py` (final per-method XAI evaluation: insertion/deletion against random baselines, expert-mask pointing game, scan-cluster statistics, Table 8 and Figure 8) to `/content`, so the notebook is self-contained.

In [ ]:
%%writefile /content/br35h_data.py
"""Br35H data utilities: tumor masks (VIA annotations), scan clusters, leakage flags, brain mask.

Verified against the public release (github.com/AhmedHamada0/Brain-Tumor-Detection-Dataset):
  * 801 annotated images (TRAIN 500 / VAL 201 / TEST 100), 931 regions:
    666 polygons, 257 ellipses (143 rotated, theta in radians), 8 circles.
  * every annotated image is byte-identical (MD5) to yes/<same name>, so masks are
    matched to the classification set by MD5, which survives renaming and de-duplication.
  * images have 111 different native sizes -> masks are rasterised at native size and then
    resized exactly like the model input (A.Resize(224,224), no aspect preservation).
"""
import hashlib, json, os, glob
import numpy as np
from PIL import Image, ImageDraw


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def _ellipse_poly(cx, cy, rx, ry, theta=0.0, n=72):
    # VIA draws with canvas ellipse(cx, cy, rx, ry, theta): rotation in image coords (y down)
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    c, s = np.cos(theta), np.sin(theta)
    x = cx + rx * np.cos(t) * c - ry * np.sin(t) * s
    y = cy + rx * np.cos(t) * s + ry * np.sin(t) * c
    return list(zip(x.tolist(), y.tolist()))


def rasterise(regions, W, H):
    m = Image.new("L", (W, H), 0)
    d = ImageDraw.Draw(m)
    for g in regions:
        sa = g["shape_attributes"]
        n = sa["name"]
        if n == "polygon":
            pts = list(zip(sa["all_points_x"], sa["all_points_y"]))
            if len(pts) >= 3:
                d.polygon(pts, fill=1)
        elif n == "ellipse":
            d.polygon(_ellipse_poly(sa["cx"], sa["cy"], sa["rx"], sa["ry"], sa.get("theta", 0.0)), fill=1)
        elif n == "circle":
            d.polygon(_ellipse_poly(sa["cx"], sa["cy"], sa["r"], sa["r"]), fill=1)
        elif n == "rect":
            d.rectangle([sa["x"], sa["y"], sa["x"] + sa["width"], sa["y"] + sa["height"]], fill=1)
        else:
            raise ValueError(f"unsupported VIA shape: {n}")
    return np.array(m, dtype=bool)


def load_br35h_masks(ann_root, size=224):
    """Returns {md5: {'mask': bool[size,size], 'mask_native': bool[H,W], 'file': str,
    'split': str, 'n_regions': int, 'area_frac': float}} for every annotated image."""
    out = {}
    for jf in sorted(glob.glob(os.path.join(ann_root, "*", "annotations_*.json"))):
        split = os.path.basename(os.path.dirname(jf))
        with open(jf) as f:
            meta = json.load(f)
        meta = meta.get("_via_img_metadata", meta)
        for rec in meta.values():
            p = os.path.join(os.path.dirname(jf), rec["filename"])
            if not os.path.exists(p):
                continue
            regs = rec["regions"]
            regs = list(regs.values()) if isinstance(regs, dict) else regs
            W, H = Image.open(p).size
            nat = rasterise(regs, W, H)
            m = np.array(Image.fromarray(nat.astype(np.uint8) * 255).resize((size, size), Image.NEAREST)) > 127
            out[_md5(p)] = dict(mask=m, mask_native=nat, file=rec["filename"], split=split,
                                n_regions=len(regs), area_frac=float(m.mean()))
    return out


def attach_masks(test_df, masks, path_col="path"):
    """Adds 'md5' and 'has_mask' columns; returns (test_df, list_of_masks_or_None aligned to rows)."""
    md5s = [_md5(p) for p in test_df[path_col]]
    test_df = test_df.copy()
    test_df["md5"] = md5s
    test_df["has_mask"] = [h in masks for h in md5s]
    aligned = [masks[h]["mask"] if h in masks else None for h in md5s]
    return test_df, aligned


# ---------------------------------------------------------------- scan clusters
def scan_clusters(paths, phash_thr=5, mirror_corr=0.95):
    """Groups images that are the same scan: identical MD5, pHash Hamming <= phash_thr,
    or a left-right mirrored copy (pHash of the mirror <= phash_thr AND pixel correlation
    of the mirror > mirror_corr AND higher than the unmirrored correlation + 0.05).
    The pixel check is needed because axial slices are roughly symmetric, so mirrored
    pHashes alone collide for unrelated scans.
    On the full Br35H yes/no set (3,000 images) this gives 624 clusters
    (385 tumor, 239 no-tumor, 0 mixed-class); 801 annotated images -> 247 clusters.
    Returns np.array of cluster ids aligned with `paths`."""
    import imagehash
    from PIL import ImageOps
    n = len(paths)
    par = list(range(n))

    def find(i):
        while par[i] != i:
            par[i] = par[par[i]]
            i = par[i]
        return i

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            par[rb] = ra

    gray = [Image.open(p).convert("L") for p in paths]
    H = np.array([np.asarray(imagehash.phash(g).hash).ravel() for g in gray], np.uint8)
    Hm = np.array([np.asarray(imagehash.phash(ImageOps.mirror(g)).hash).ravel() for g in gray], np.uint8)
    md5s = {}
    for i, p in enumerate(paths):
        h = _md5(p)
        if h in md5s:
            union(md5s[h], i)
        else:
            md5s[h] = i
    small = None

    def z(i, mirror=False):
        nonlocal small
        if small is None:
            small = {}
        k = (i, mirror)
        if k not in small:
            g = ImageOps.mirror(gray[i]) if mirror else gray[i]
            a = np.asarray(g.resize((128, 128)), np.float64)
            small[k] = (a - a.mean()) / (a.std() + 1e-8)
        return small[k]

    for i in range(n):
        d = (H[i + 1:] != H[i]).sum(1)
        for j in np.where(d <= phash_thr)[0] + i + 1:
            union(i, j)
        dm = (Hm[i + 1:] != H[i]).sum(1)
        for j in np.where(dm <= phash_thr)[0] + i + 1:
            if find(i) == find(j):
                continue
            r_m = float((z(i, True) * z(j)).mean())
            r_d = float((z(i) * z(j)).mean())
            if r_m > mirror_corr and r_m > r_d + 0.05:
                union(i, j)
    return np.array([find(i) for i in range(n)])


def leakage_flags(test_paths, train_paths):
    """For each test image: does any training (or validation) image belong to the same scan
    cluster? Returns (cluster_ids_for_test, bool array)."""
    allp = list(train_paths) + list(test_paths)
    cl = scan_clusters(allp)
    tr = set(cl[: len(train_paths)])
    te = cl[len(train_paths):]
    return te, np.array([c in tr for c in te])


# ---------------------------------------------------------------- brain mask
def brain_mask(img_u8):
    """Same definition as the notebook's _brain_mask (Otsu, largest component, dilated)."""
    import cv2
    gray = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY) if img_u8.ndim == 3 else img_u8
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n_cc, lab, stats, _ = cv2.connectedComponentsWithStats(th, connectivity=8)
    if n_cc <= 1:
        return np.ones_like(gray, dtype=bool)
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    m = (lab == largest).astype(np.uint8)
    return cv2.dilate(m, np.ones((5, 5), np.uint8), iterations=2).astype(bool)

In [ ]:
%%writefile /content/xai_faithfulness_eval.py
"""
Final XAI faithfulness and localisation evaluation for GCAF-CEViT (Br35H).
Responds to the reviewer comment on Section III.D / Table 8 / Figure 8.

For every attribution method, and for two random-attribution baselines:
  * Insertion AUC (higher = better) and Deletion AUC (lower = better), Petsiuk et al. 2018 (RISE)
  * Gap = Insertion - Deletion, and the gap minus the in-brain random gap
  * Pointing game against the expert Br35H tumor polygons/ellipses (not a brain mask),
    with 15-px tolerance (Zhang et al. 2016), the chance level for a random in-brain
    point, and energy-in-mask with its chance level.
Statistics treat copies of the same scan as ONE unit: per-image scores are averaged within
scan clusters, then Wilcoxon signed-rank (Holm-corrected) and cluster bootstrap CIs are
computed over clusters. (Br35H contains heavy duplication: 3,000 images -> 624 scan clusters.)

Strata: All / Tumor / No tumor / Correctly classified / Test images with no copy in training.

Usage inside the notebook: see notebook_xai_final_cells.py (cells A-E).
Self-test without data:  python xai_faithfulness_eval.py
"""
import json
import os
import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import wilcoxon, rankdata
from scipy.ndimage import binary_dilation

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

CFG = dict(
    img_size=224,
    n_steps=50,               # 2% of pixels per step; 100 is finer but 2x slower
    baseline="blur",          # primary: "blur"; sensitivity: "black" or "brain_mean"
    blur_kernel=31,
    blur_sigma=10.0,
    n_random_seeds=3,
    pointing_tolerance=15,
    target="pred",            # explain the PREDICTED class (notebook cell 51 rule)
    n_boot=2000,
    seed=0,
    batch=51,
)
TUMOR, NO_TUMOR = 1, 0
RANDOM_U, RANDOM_B = "Random (uniform)", "Random (in-brain)"


# ------------------------------------------------------------------ helpers
def gaussian_blur(x, k=31, sigma=10.0):
    ax = torch.arange(k, dtype=x.dtype, device=x.device) - (k - 1) / 2
    g = torch.exp(-(ax ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    C = x.shape[1]
    x = F.conv2d(F.pad(x, (k // 2, k // 2, 0, 0), mode="reflect"), g.view(1, 1, 1, k).repeat(C, 1, 1, 1), groups=C)
    x = F.conv2d(F.pad(x, (0, 0, k // 2, k // 2), mode="reflect"), g.view(1, 1, k, 1).repeat(C, 1, 1, 1), groups=C)
    return x


def black_like(x):
    """Pixel value 0 (black) expressed in ImageNet-normalised space.
    torch.zeros_like(x) would be ImageNet-mean GREY, not black."""
    m = torch.tensor(IMAGENET_MEAN, dtype=x.dtype, device=x.device).view(1, 3, 1, 1)
    s = torch.tensor(IMAGENET_STD, dtype=x.dtype, device=x.device).view(1, 3, 1, 1)
    return ((0 - m) / s).expand_as(x).clone()


def make_baseline(x, brain=None, cfg=CFG):
    b = cfg["baseline"]
    if b == "blur":
        return gaussian_blur(x, cfg["blur_kernel"], cfg["blur_sigma"])
    if b == "black":
        return black_like(x)
    if b == "brain_mean":
        bm = torch.as_tensor(brain, dtype=x.dtype, device=x.device)[None, None]
        mu = (x * bm).sum((2, 3), keepdim=True) / bm.sum().clamp(min=1)
        return x * (1 - bm) + mu * bm
    raise ValueError(b)


@torch.no_grad()
def probs(forward, x):
    return F.softmax(forward(x), dim=1)


@torch.no_grad()
def causal_curves(forward, x, sal, target, base, cfg=CFG):
    """Returns (insertion_curve, deletion_curve), each (n_steps+1,)."""
    H, W = sal.shape
    n = H * W
    order = torch.as_tensor(np.argsort(-sal.reshape(-1), kind="stable").copy(), device=x.device)
    steps = cfg["n_steps"]
    counts = [int(round(i * n / steps)) for i in range(steps + 1)]
    rank = torch.empty(n, device=x.device, dtype=torch.long)
    rank[order] = torch.arange(n, device=x.device)
    out = {}
    for mode in ("insertion", "deletion"):
        start, finish = (base, x) if mode == "insertion" else (x, base)
        vals = []
        for j in range(0, len(counts), cfg["batch"]):
            cs = torch.tensor(counts[j:j + cfg["batch"]], device=x.device)
            m = (rank[None, :] < cs[:, None]).float().view(-1, 1, H, W)
            imgs = start * (1 - m) + finish * m
            vals.append(probs(forward, imgs)[:, target].cpu())
        out[mode] = torch.cat(vals).numpy()
    return out["insertion"], out["deletion"]


def curve_auc(c):
    xs = np.linspace(0, 1, len(c))
    f = getattr(np, "trapezoid", None) or np.trapz
    return float(f(c, xs))


def to_hw(sal, size):
    sal = np.asarray(sal, dtype=np.float64)
    if sal.ndim == 3:
        sal = sal.sum(0)
    if sal.shape != (size, size):
        t = torch.as_tensor(sal)[None, None].float()
        sal = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)[0, 0].numpy()
    return np.nan_to_num(sal)


def random_map(shape, rng, brain=None):
    r = rng.random(shape)
    if brain is not None:
        r = np.where(brain, r, r - 2.0)   # all brain pixels ranked before background
    return r


def pointing(sal, mask, brain, tol):
    """Hit if the argmax is inside the tumor mask dilated by tol px.
    chance = P(uniform random in-brain point lands in the dilated mask)."""
    md = binary_dilation(mask, iterations=tol) if tol > 0 else mask
    y, x = np.unravel_index(np.argmax(sal), sal.shape)
    hit = bool(md[y, x])
    chance = float((md & brain).sum() / max(brain.sum(), 1))
    s = np.clip(sal, 0, None)
    energy = float(s[mask].sum() / (s.sum() + 1e-12))
    energy_chance = float(mask.sum() / max(brain.sum(), 1))
    return hit, chance, energy, energy_chance


# --------------------------------------------------------------- statistics
def cluster_means(values, clusters):
    v, c = np.asarray(values, float), np.asarray(clusters)
    u = np.unique(c)
    return np.array([v[c == k].mean() for k in u])


def cluster_boot_ci(values, clusters, n_boot, rng):
    v, c = np.asarray(values, float), np.asarray(clusters)
    if len(v) == 0:
        return (np.nan, np.nan)
    u = np.unique(c)
    groups = [v[c == k] for k in u]
    sums = np.array([g.sum() for g in groups]); lens = np.array([len(g) for g in groups])
    idx = rng.integers(0, len(u), (n_boot, len(u)))
    m = sums[idx].sum(1) / lens[idx].sum(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def holm(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    adj = np.empty(n); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (n - r) * p[i]); adj[i] = min(1.0, run)
    return adj


def paired_cluster_test(a, b, clusters, alternative):
    """Wilcoxon on cluster-level means of (a - b). Returns (p, rank-biserial, n_clusters)."""
    d = cluster_means(np.asarray(a) - np.asarray(b), clusters)
    nz = d[np.abs(d) > 1e-12]
    if len(nz) < 6:
        return np.nan, np.nan, len(d)
    _, p = wilcoxon(nz, alternative=alternative)
    r = rankdata(np.abs(nz))
    return float(p), float((r[nz > 0].sum() - r[nz < 0].sum()) / r.sum()), len(d)


# --------------------------------------------------------------------- main
def evaluate(forward, items, explainers, cfg=CFG, out_dir="xai_results", log_every=10):
    """
    forward   : callable x -> logits (B, 2)   (use the notebook's `wrapped`)
    items     : list of dicts with keys
                x (1,3,S,S) normalised tensor on the model device, label, pred, prob (p_tumor),
                id, cluster (scan-cluster id), leak (bool: same scan in train/val),
                brain (bool S x S), mask (bool S x S tumor mask or None)
    explainers: {name: fn(x, target) -> 2-D (or C x H x W) array}
    """
    os.makedirs(out_dir, exist_ok=True)
    rng = np.random.default_rng(cfg["seed"])
    S = cfg["img_size"]
    names = list(explainers) + [RANDOM_U, RANDOM_B]
    rows = []
    for k, it in enumerate(items):
        x, y, pred = it["x"], int(it["label"]), int(it["pred"])
        t = pred if cfg["target"] == "pred" else y
        brain = it["brain"]
        base = make_baseline(x, brain, cfg)
        maps = {}
        for n, fn in explainers.items():
            try:
                maps[n] = to_hw(fn(x, t), S)
            except Exception as e:  # keep going, but record the failure
                print(f"[{it['id']}] {n} failed: {e}")
                maps[n] = None
        # tumor-class maps for the pointing game when the prediction is not 'tumor' (FN)
        pg_maps = maps
        if y == TUMOR and it.get("mask") is not None and t != TUMOR:
            pg_maps = {}
            for n, fn in explainers.items():
                try:
                    pg_maps[n] = to_hw(fn(x, TUMOR), S)
                except Exception:
                    pg_maps[n] = None
        for n in names:
            r = dict(id=it["id"], method=n, label=y, pred=pred, target=t, prob_tumor=float(it["prob"]),
                     cluster=str(it["cluster"]), leak=bool(it["leak"]), has_mask=it.get("mask") is not None)
            if n in (RANDOM_U, RANDOM_B):
                ins, dele, hits, en = [], [], [], []
                for _ in range(cfg["n_random_seeds"]):
                    sal = random_map((S, S), rng, brain if n == RANDOM_B else None)
                    ci, cd = causal_curves(forward, x, sal, t, base, cfg)
                    ins.append(curve_auc(ci)); dele.append(curve_auc(cd))
                    if y == TUMOR and r["has_mask"]:
                        h, ch, e, ech = pointing(sal, it["mask"], brain, cfg["pointing_tolerance"])
                        hits.append(h); en.append(e)
                r.update(ins=float(np.mean(ins)), dele=float(np.mean(dele)))
                if hits:
                    r.update(hit=float(np.mean(hits)), chance=ch, energy=float(np.mean(en)), energy_chance=ech)
            else:
                sal = maps[n]
                if sal is None:
                    continue
                ci, cd = causal_curves(forward, x, sal, t, base, cfg)
                r.update(ins=curve_auc(ci), dele=curve_auc(cd))
                if y == TUMOR and r["has_mask"] and pg_maps.get(n) is not None:
                    h, ch, e, ech = pointing(pg_maps[n], it["mask"], brain, cfg["pointing_tolerance"])
                    r.update(hit=float(h), chance=ch, energy=e, energy_chance=ech)
            r["gap"] = r["ins"] - r["dele"]
            rows.append(r)
        if log_every and (k + 1) % log_every == 0:
            print(f"  {k + 1}/{len(items)} images done")
    import pandas as pd
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out_dir, "xai_per_image.csv"), index=False)
    summary = summarise(df, list(explainers), cfg, rng)
    with open(os.path.join(out_dir, "xai_summary.json"), "w") as f:
        json.dump({"config": cfg, "summary": summary}, f, indent=2, default=float)
    with open(os.path.join(out_dir, "table8.tex"), "w") as f:
        f.write(latex_table8(summary))
    with open(os.path.join(out_dir, "table8.md"), "w") as f:
        f.write(markdown_table8(summary))
    return df, summary


def summarise(df, methods, cfg, rng):
    strata = {
        "All": df.index == df.index,
        "Tumor": df.label == TUMOR,
        "No tumor": df.label == NO_TUMOR,
        "Correct": df.label == df.pred,
        "No train copy": ~df.leak,
    }
    out = {}
    names = methods + [RANDOM_U, RANDOM_B]
    for s, sel in strata.items():
        out[s] = {}
        sub = df[sel]
        refB = sub[sub.method == RANDOM_B].set_index("id")
        refU = sub[sub.method == RANDOM_U].set_index("id")
        raw, keys = [], []
        for n in names:
            d = sub[sub.method == n].set_index("id")
            if len(d) == 0:
                continue
            e = dict(n_images=int(len(d)), n_clusters=int(d.cluster.nunique()))
            for col in ("ins", "dele", "gap"):
                cm = cluster_means(d[col], d.cluster)
                e[f"{col}_mean"] = float(d[col].mean())
                e[f"{col}_sd_clusters"] = float(cm.std(ddof=1)) if len(cm) > 1 else float("nan")
                e[f"{col}_ci"] = cluster_boot_ci(d[col], d.cluster, cfg["n_boot"], rng)
            ids = d.index.intersection(refB.index)
            e["gap_minus_random_brain"] = float((d.loc[ids, "gap"] - refB.loc[ids, "gap"]).mean()) if len(ids) else float("nan")
            if n in methods:
                for ref, rname in ((refB, "brain"), (refU, "uniform")):
                    ids = d.index.intersection(ref.index)
                    cl = d.loc[ids, "cluster"]
                    p_i, r_i, nc = paired_cluster_test(d.loc[ids, "ins"], ref.loc[ids, "ins"], cl, "greater")
                    p_d, r_d, _ = paired_cluster_test(d.loc[ids, "dele"], ref.loc[ids, "dele"], cl, "less")
                    e[f"ins_rbc_vs_{rname}"], e[f"del_rbc_vs_{rname}"] = r_i, r_d
                    raw += [p_i, p_d]; keys += [(n, f"ins_p_holm_vs_{rname}"), (n, f"del_p_holm_vs_{rname}")]
            out[s][n] = e
        valid = [i for i, p in enumerate(raw) if not np.isnan(p)]
        adj = holm([raw[i] for i in valid]) if valid else []
        for k, (n, key) in enumerate(keys):
            out[s][n][key] = float(adj[valid.index(k)]) if k in valid else None
    # pointing game on tumor images with expert masks
    pg = {}
    pgdf = df[(df.label == TUMOR) & df.has_mask & df.hit.notna()] if "hit" in df else df.iloc[0:0]
    raw, keys = [], []
    for n in names:
        d = pgdf[pgdf.method == n]
        if len(d) == 0:
            pg[n] = dict(n_images=0)
            continue
        e = dict(n_images=int(len(d)), n_clusters=int(d.cluster.nunique()),
                 hit_rate=float(d.hit.mean()), hit_ci=cluster_boot_ci(d.hit, d.cluster, cfg["n_boot"], rng),
                 chance=float(d.chance.mean()), energy=float(d.energy.mean()),
                 energy_chance=float(d.energy_chance.mean()),
                 energy_ci=cluster_boot_ci(d.energy, d.cluster, cfg["n_boot"], rng))
        if n in methods:
            p, r, _ = paired_cluster_test(d.hit, d.chance, d.cluster, "greater")
            pe, re_, _ = paired_cluster_test(d.energy, d.energy_chance, d.cluster, "greater")
            e.update(hit_rbc=r, energy_rbc=re_)
            raw += [p, pe]; keys += [(n, "hit_p_holm_vs_chance"), (n, "energy_p_holm_vs_chance")]
        pg[n] = e
    valid = [i for i, p in enumerate(raw) if not np.isnan(p)]
    adj = holm([raw[i] for i in valid]) if valid else []
    for k, (n, key) in enumerate(keys):
        pg[n][key] = float(adj[valid.index(k)]) if k in valid else None
    out["Pointing game (tumor images with expert mask)"] = pg
    return out


# ------------------------------------------------------------------ tables
def _fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "--"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def markdown_table8(s):
    T, N, A = s["Tumor"], s["No tumor"], s["All"]
    pg = s["Pointing game (tumor images with expert mask)"]
    L = ["| Method | Ins (tumor) | Del (tumor) | Gap (tumor) | Ins (no tumor) | Del (no tumor) | "
         "Gap - random (all) | p ins / del vs in-brain random (all) | Pointing hit (n) | Chance | Energy in mask (chance) |",
         "|" + "---|" * 11]
    for n in A:
        t, nt, a, p = T.get(n, {}), N.get(n, {}), A[n], pg.get(n, {})
        f = lambda e, k: f"{e[k]:.3f}" if k in e else "--"
        if not p.get("n_images"):
            hit = "--"
        elif n in (RANDOM_U, RANDOM_B):
            hit = f"{100 * p['hit_rate']:.1f}% (n={p['n_images']})"
        else:
            hit = f"{100 * p['hit_rate']:.1f}% ({int(round(p['hit_rate'] * p['n_images']))}/{p['n_images']})"
        ch = f"{100 * p['chance']:.1f}%" if p.get("n_images") else "--"
        en = f"{p['energy']:.3f} ({p['energy_chance']:.3f})" if p.get("n_images") else "--"
        pv = f"{_fmt_p(a.get('ins_p_holm_vs_brain'))} / {_fmt_p(a.get('del_p_holm_vs_brain'))}" if "ins_p_holm_vs_brain" in a else "ref."
        L.append(f"| {n} | {f(t, 'ins_mean')} | {f(t, 'dele_mean')} | {f(t, 'gap_mean')} | {f(nt, 'ins_mean')} | "
                 f"{f(nt, 'dele_mean')} | {a['gap_minus_random_brain']:+.3f} | {pv} | {hit} | {ch} | {en} |")
    nA = next(iter(A.values()))
    L.append(f"\nImages: {nA['n_images']} ({nA['n_clusters']} scan clusters). Pointing game: tumor images with expert masks. "
             "Stats are over scan clusters (Wilcoxon, Holm).")
    return "\n".join(L) + "\n"


def latex_table8(s):
    T, N, A = s["Tumor"], s["No tumor"], s["All"]
    pg = s["Pointing game (tumor images with expert mask)"]
    out = [r"\begin{table*}[t]\centering",
           r"\caption{Per-method causal faithfulness (insertion/deletion AUC, blur baseline) and tumor localisation "
           r"(pointing game against the Br35H expert annotations, 15-px tolerance). Statistics are computed over scan "
           r"clusters; $p$-values: one-sided Wilcoxon vs.\ in-brain random attribution, Holm-corrected.}",
           r"\label{tab:xai}",
           r"\begin{tabular}{lcccccccc}\hline",
           r"Method & Ins$\uparrow$ (T) & Del$\downarrow$ (T) & Ins$\uparrow$ (NT) & Del$\downarrow$ (NT) & "
           r"Gap$-$rand & $p_{ins}/p_{del}$ & Pointing (n) & Chance \\ \hline"]
    for n in A:
        t, nt, a, p = T.get(n, {}), N.get(n, {}), A[n], pg.get(n, {})
        f = lambda e, k: f"{e[k]:.3f}" if k in e else "--"
        hit = f"{100 * p['hit_rate']:.1f}\\% ({p['n_images']})" if p.get("n_images") else "--"
        ch = f"{100 * p['chance']:.1f}\\%" if p.get("n_images") else "--"
        pv = f"{_fmt_p(a.get('ins_p_holm_vs_brain'))}/{_fmt_p(a.get('del_p_holm_vs_brain'))}" if "ins_p_holm_vs_brain" in a else "ref."
        out.append(f"{n} & {f(t, 'ins_mean')} & {f(t, 'dele_mean')} & {f(nt, 'ins_mean')} & {f(nt, 'dele_mean')} & "
                   f"{a['gap_minus_random_brain']:+.3f} & {pv.replace('<', '$<$')} & {hit} & {ch} \\\\")
    out += [r"\hline\end{tabular}\end{table*}"]
    return "\n".join(out) + "\n"


# --------------------------------------------------------------- figure 8
def figure8(items, explainers, case_idx, out_path, class_names=("No tumor", "Tumor"), cfg=CFG):
    """case_idx: {'TP': [...], 'FN': [...], 'TN': [...], 'FP': [...]} indices into items.
    Rows = cases, columns = original + each method; expert tumor mask outlined in green."""
    import matplotlib
    import matplotlib.pyplot as plt
    order = [(k, i) for k in ("TP", "FN", "TN", "FP") for i in case_idx.get(k, [])]
    m = list(explainers)
    fig, axes = plt.subplots(len(order), len(m) + 1, figsize=(2.3 * (len(m) + 1), 2.9 * len(order)), squeeze=False)
    mean = np.array(IMAGENET_MEAN)[:, None, None]; std = np.array(IMAGENET_STD)[:, None, None]
    for r, (kind, i) in enumerate(order):
        it = items[i]
        img = np.clip(it["x"][0].detach().cpu().numpy() * std + mean, 0, 1).transpose(1, 2, 0)
        t = int(it["pred"]) if cfg["target"] == "pred" else int(it["label"])
        conf = it["prob"] if t == 1 else 1 - it["prob"]
        ax = axes[r, 0]; ax.imshow(img)
        if it.get("mask") is not None:
            ax.contour(it["mask"], levels=[0.5], colors="lime", linewidths=1.0)
        ax.set_title(f"{kind}: true {class_names[it['label']]}\npred {class_names[it['pred']]} ({conf:.2f})", fontsize=7)
        for c, n in enumerate(m, start=1):
            ax = axes[r, c]
            try:
                sal = to_hw(explainers[n](it["x"], t), cfg["img_size"])
                # display only: positive evidence for the target class, clipped at the 99th percentile
                # (signed IG/SHAP maps would otherwise put zero at mid-colour and wash out the panel)
                sal = np.clip(sal, 0, None)
                hi = np.percentile(sal, 99) if sal.max() > 0 else 1.0
                sal = np.clip(sal / (hi + 1e-12), 0, 1)
                if n in ("Integrated Gradients", "SHAP (GradientSHAP)"):
                    from scipy.ndimage import gaussian_filter
                    sal = gaussian_filter(sal, 2); sal = sal / (sal.max() + 1e-12)
                ax.imshow(img); ax.imshow(sal, cmap="jet", alpha=0.45)
                if it.get("mask") is not None:
                    ax.contour(it["mask"], levels=[0.5], colors="lime", linewidths=1.0)
            except Exception as e:
                ax.text(0.5, 0.5, "failed", ha="center")
            if r == 0:
                ax.set_title(n, fontsize=8)
        for ax in axes[r]:
            ax.axis("off")
    plt.tight_layout(h_pad=1.5)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


# ---------------------------------------------------------------- adapters
def build_explainers(model, forward, cfg=CFG, lime_samples=500, scorecam_batch=64, which=None):
    """Adapters for the notebook's HybridModel (cell 16):
         model.cnn.cbam        -> CBAM-refined (B,1280,7,7) map  (Grad-CAM++, Score-CAM)
         model.vit.vit.blocks  -> timm ViT blocks                (attention rollout, ViT branch only)
       `forward` must return logits only (the notebook's `wrapped`).
       Attribution sign convention: positive = evidence FOR the target class."""
    from pytorch_grad_cam import GradCAMPlusPlus, ScoreCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from captum.attr import IntegratedGradients, Occlusion, Lime, GradientShap
    from captum._utils.models.linear_model import SkLearnLinearRegression

    layer = [model.cnn.cbam]
    ex = {}

    def f_gcpp(x, t):
        with GradCAMPlusPlus(model=forward, target_layers=layer) as cam:
            return cam(input_tensor=x, targets=[ClassifierOutputTarget(t)])[0]
    ex["Grad-CAM++"] = f_gcpp

    def f_scam(x, t):
        with ScoreCAM(model=forward, target_layers=layer) as cam:
            cam.batch_size = scorecam_batch
            return cam(input_tensor=x, targets=[ClassifierOutputTarget(t)])[0]
    ex["Score-CAM"] = f_scam

    def f_rollout(x, t):
        blocks = model.vit.vit.blocks
        att, hooks = [], []
        for b in blocks:
            b.attn.fused_attn = False
            hooks.append(b.attn.attn_drop.register_forward_hook(lambda _m, _i, o: att.append(o.detach())))
        try:
            with torch.no_grad():
                forward(x)
        finally:
            for h in hooks:
                h.remove()
        R = torch.eye(att[0].shape[-1], device=att[0].device)
        for A in att:
            A = A[0].mean(0)
            A = A + torch.eye(A.shape[-1], device=A.device)
            A = A / A.sum(-1, keepdim=True)
            R = A @ R
        g = int((R.shape[-1] - 1) ** 0.5)
        return R[0, 1:].reshape(g, g).cpu().numpy()   # class-agnostic: same map for both targets
    ex["Attention rollout (ViT)"] = f_rollout

    ig = IntegratedGradients(forward)
    def f_ig(x, t):
        a = ig.attribute(x, baselines=gaussian_blur(x, cfg["blur_kernel"], cfg["blur_sigma"]),
                         target=t, n_steps=32, internal_batch_size=16)
        return a[0].sum(0).detach().cpu().numpy()
    ex["Integrated Gradients"] = f_ig

    gs = GradientShap(forward)
    def f_shap(x, t):
        base = torch.cat([gaussian_blur(x, cfg["blur_kernel"], cfg["blur_sigma"]), black_like(x)])
        a = gs.attribute(x, baselines=base, target=t, n_samples=32, stdevs=0.05)
        return a[0].sum(0).detach().cpu().numpy()
    ex["SHAP (GradientSHAP)"] = f_shap

    occ = Occlusion(forward)
    def f_occ(x, t):
        a = occ.attribute(x, target=t, sliding_window_shapes=(3, 32, 32), strides=(3, 16, 16),
                          baselines=gaussian_blur(x, cfg["blur_kernel"], cfg["blur_sigma"]), perturbations_per_eval=32)
        return a[0].sum(0).detach().cpu().numpy()
    ex["Occlusion"] = f_occ

    lime = Lime(forward, interpretable_model=SkLearnLinearRegression())
    def f_lime(x, t):
        from skimage.segmentation import slic
        mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
        img = (x[0].detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        seg = slic(img, n_segments=50, compactness=10, start_label=0)
        fm = torch.as_tensor(seg, device=x.device)[None, None].expand(1, 3, -1, -1)
        a = lime.attribute(x, target=t, feature_mask=fm, n_samples=lime_samples, perturbations_per_eval=32,
                           baselines=gaussian_blur(x, cfg["blur_kernel"], cfg["blur_sigma"]))
        return a[0, 0].detach().cpu().numpy()
    ex["LIME"] = f_lime

    if which:
        ex = {k: v for k, v in ex.items() if k in which}
    return ex


# ---------------------------------------------------------------- self-test
if __name__ == "__main__":
    torch.manual_seed(0)
    S = 224

    class Toy(torch.nn.Module):
        """Tumor logit is driven ONLY by the brightness inside a lesion region."""
        def forward(self, x):
            z = 12 * (x[:, :, 95:125, 95:125].mean((1, 2, 3)) - 1.0)
            return torch.stack([-z, z], 1)

    toy = Toy()
    items = []
    for i in range(16):
        x = black_like(torch.zeros(1, 3, S, S))
        x[:, :, 30:194, 30:194] = 0.0
        lab = i % 2
        if lab:
            x[:, :, 95:125, 95:125] = 2.0
        brain = np.zeros((S, S), bool); brain[30:194, 30:194] = True
        mask = np.zeros((S, S), bool); mask[95:125, 95:125] = True
        p = float(F.softmax(toy(x), 1)[0, 1])
        items.append(dict(x=x, label=lab, pred=int(p > 0.5), prob=p, id=str(i), cluster=str(i // 2),
                          leak=False, brain=brain, mask=mask if lab else None))

    def oracle(x, t):
        s = np.zeros((S, S)); s[95:125, 95:125] = 1.0
        return s + 1e-3 * np.random.rand(S, S)

    def anti(x, t):
        s = np.ones((S, S)); s[95:125, 95:125] = 0.0
        return s + 1e-3 * np.random.rand(S, S)

    cfg = dict(CFG, n_steps=20, n_random_seeds=2, n_boot=300, baseline="black")
    df, s = evaluate(toy, items, {"Oracle": oracle, "Anti-oracle": anti}, cfg=cfg, out_dir="/tmp/xai_selftest", log_every=0)
    T = s["Tumor"]; pg = s["Pointing game (tumor images with expert mask)"]
    print(open("/tmp/xai_selftest/table8.md").read())
    assert T["Oracle"]["ins_mean"] > T[RANDOM_B]["ins_mean"] and T["Oracle"]["dele_mean"] < T[RANDOM_B]["dele_mean"]
    assert T["Anti-oracle"]["dele_mean"] > T["Oracle"]["dele_mean"]
    assert T["Oracle"]["gap_mean"] > 0.3 and T["Anti-oracle"]["gap_mean"] < 0
    assert pg["Oracle"]["hit_rate"] == 1.0 and pg["Anti-oracle"]["hit_rate"] == 0.0
    assert abs(pg[RANDOM_B]["hit_rate"] - pg[RANDOM_B]["chance"]) < 0.25
    print("self-test passed")

In [ ]:
import sys, importlib
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import br35h_data, xai_faithfulness_eval
importlib.reload(br35h_data); importlib.reload(xai_faithfulness_eval)
print('XAI helper modules ready')

## Cell 5 — Configuration

Most of this is fine as-is. `DATASET_PATH` is set automatically from Cell 3 — only touch it if you skipped that cell.

In [ ]:
class Config:
    # ──────────────────────────────────────────────────────────────────────
    # Set automatically by the Kaggle-download cell above. Only edit this by
    # hand if you skipped that cell and already have the data somewhere else
    # (point it at the folder that directly contains 'yes' and 'no').
    DATASET_PATH = globals().get('DATASET_PATH', '/content/drive/MyDrive/Br35H/')
    # ──────────────────────────────────────────────────────────────────────

    IMG_SIZE      = 224
    BATCH_SIZE    = 16          # lower than a typical default — dual CNN+ViT branches are memory-hungry
    NUM_EPOCHS    = 50          # common maximum budget for full model and all ablations
    LR            = 1e-5
    WEIGHT_DECAY  = 1e-5
    WARMUP_EPOCHS = 3
    LABEL_SMOOTH  = 0.05
    GRAD_CLIP     = 1.0
    DROPOUT       = 0.4
    FUSION_DIM    = 512

    TRAIN_RATIO   = 0.70
    VAL_RATIO     = 0.15
    TEST_RATIO    = 0.15
    SEED          = 42

    # True = lighter backbones + smaller batch, recommended on a free Colab T4.
    LOW_MEMORY_MODE = True
    CNN_BACKBONE  = 'efficientnet_b0'       if LOW_MEMORY_MODE else 'tf_efficientnetv2_s'
    VIT_BACKBONE  = 'vit_small_patch16_224' if LOW_MEMORY_MODE else 'vit_base_patch16_224'

    # K-Fold CV and the ablation study each retrain the model multiple times —
    # that's hours on a free GPU. Keep False until your main run works end-to-end.
    RUN_HEAVY_CELLS = True
    # Split by SCAN cluster (exact, rescaled and mirrored copies of the same slice share a
    # group). With file-name groups (False) 397/443 test images share a scan with train.
    SCAN_GROUP_SPLIT = True
    # Final XAI evaluation (Cell 15c). XAI_MAX_IMAGES=None -> whole test set (~1 h on a T4);
    # an int -> all annotated tumor images + a seeded no-tumor subset of the same size.
    XAI_MAX_IMAGES = None
    XAI_RUN_BLACK_SENSITIVITY = False
    # Fair comparison protocol: same maximum budget, warm-up, early stopping,
    # and validation-AUC checkpoint criterion for all model variants.
    KFOLD_EPOCHS    = NUM_EPOCHS
    ABLATION_EPOCHS = NUM_EPOCHS
    EARLY_STOPPING_PATIENCE = 10
    CHECKPOINT_METRIC = "val_auc"
    CHECKPOINT_MODE = "max"

    USE_MIXUP     = False
    USE_CUTMIX    = False
    MIXUP_ALPHA   = 0.2

    NUM_CLASSES   = 2
    CLASS_NAMES   = ['no_tumor', 'tumor']   # placeholder — overwritten after the dataset scan

    CHECKPOINT    = '/content/drive/MyDrive/brain_tumor_results/checkpoints/'
    RESULTS       = '/content/drive/MyDrive/brain_tumor_results/results/'

CFG = Config()

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = True

seed_all(CFG.SEED)
os.makedirs(CFG.CHECKPOINT, exist_ok=True)
os.makedirs(CFG.RESULTS,    exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Catch a bad path HERE, with a clear message, instead of deep inside training ──
if not os.path.isdir(CFG.DATASET_PATH):
    raise FileNotFoundError(
        f"DATASET_PATH does not exist: {CFG.DATASET_PATH}\n"
        f"Fix: in the Colab Files pane, browse to /content/drive/MyDrive/, find the "
        f"folder that directly contains your Br35H class subfolders (e.g. 'yes' and "
        f"'no'), and paste its exact path into Config.DATASET_PATH above."
    )

_EXCLUDE_DIRS = {'pred', 'predictions', '.ipynb_checkpoints'}
subfolders = [d for d in os.listdir(CFG.DATASET_PATH)
              if os.path.isdir(os.path.join(CFG.DATASET_PATH, d))
              and d.lower() not in _EXCLUDE_DIRS]
if len(subfolders) < 2:
    raise FileNotFoundError(
        f"Expected at least 2 class subfolders inside {CFG.DATASET_PATH}, found {subfolders}.\n"
        f"Br35H should have folders like 'yes' and 'no' directly inside DATASET_PATH "
        f"(not nested inside an extra wrapper folder)."
    )

print("✅ Configuration set!")
print(f"   Dataset path     : {CFG.DATASET_PATH}")
print(f"   Class subfolders : {subfolders}")
print(f"   Low-memory mode  : {CFG.LOW_MEMORY_MODE}")
print(f"   CNN backbone     : {CFG.CNN_BACKBONE}")
print(f"   ViT backbone     : {CFG.VIT_BACKBONE}")
print(f"   Device           : {DEVICE}")

## Cell 6 — Data Pipeline (Explore → Split → Augment → DataLoaders)

In [ ]:
# ── 5a: Explore dataset ─────────────────────────────────────────────────────
VALID_EXT = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif')

def explore_dataset(path):
    records, counts = [], {}
    for cls in sorted(os.listdir(path)):
        d = os.path.join(path, cls)
        if not os.path.isdir(d):
            continue
        if cls.lower() in {'pred', 'predictions', '.ipynb_checkpoints'}:
            continue  # not a class — extra unlabeled images shipped with Br35H
        imgs = [f for f in os.listdir(d) if f.lower().endswith(VALID_EXT)]
        if len(imgs) == 0:
            continue
        counts[cls] = len(imgs)
        for f in imgs:
            records.append({'path': os.path.join(d, f), 'label': cls})
    df = pd.DataFrame(records)

    if len(df) == 0:
        raise FileNotFoundError(
            f"No images found under {path}. Checked extensions: {VALID_EXT}. "
            f"Confirm the class folders actually contain image files directly "
            f"(not a further nested subfolder)."
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    col = plt.cm.viridis(np.linspace(0.2, 0.8, len(counts)))
    axes[0].bar(counts.keys(), counts.values(), color=col)
    axes[0].set_title('Class Distribution — Br35H', fontweight='bold')
    for i, (k, v) in enumerate(counts.items()):
        axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')
    axes[1].pie(counts.values(), labels=counts.keys(), autopct='%1.1f%%', colors=col)
    axes[1].set_title('Class Percentage', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}class_distribution.png', dpi=150)
    plt.show()
    print(f"\n📊 Total samples : {len(df)}")
    print(f"📊 Class counts  : {counts}")
    return df

df = explore_dataset(CFG.DATASET_PATH)

# ── 5b: Data-leakage audit — Step 1: EXACT duplicates (MD5) ────────────────
# Important because earlier runs of this notebook achieved near-perfect AUC.
import hashlib

def file_hash(path, chunk=8192):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

df['file_hash'] = df['path'].apply(file_hash)
exact_dupes = df[df.duplicated('file_hash', keep=False)].sort_values('file_hash')
print(f"Exact (MD5) duplicate images found: {len(exact_dupes)}")
if len(exact_dupes):
    print(exact_dupes[['path', 'label', 'file_hash']].head(20).to_string())
    initial_len = len(df)
    df = df.drop_duplicates(subset='file_hash', keep='first').reset_index(drop=True)
    print(f"Removed {initial_len - len(df)} exact duplicates. New dataset size: {len(df)}")
else:
    print("✅ No exact duplicate images detected.")

# ── 5c: Data-leakage audit — Step 2: NEAR duplicates (pHash) ───────────────
# MD5 only catches byte-identical files. A resized, recompressed, or lightly
# cropped copy of the same scan has a DIFFERENT MD5 but a near-identical
# perceptual hash, and is just as capable of leaking train/test information.
# We compare perceptual hashes pairwise via Hamming distance rather than
# exact string matching, so this catches "almost the same image" too.
def _phash_or_none(path):
    try:
        return imagehash.phash(Image.open(path).convert('RGB'))
    except Exception as e:
        print(f"⚠️  Could not hash {path}: {e}")
        return None

df['phash'] = df['path'].apply(_phash_or_none)
_valid = df['phash'].notna()

PHASH_HAMMING_THRESHOLD = 5   # <=5 bits different out of 64 ≈ visually near-identical
PHASH_MAX_N_FOR_PAIRWISE = 8000  # guard against O(n²) blowup on huge datasets

near_dup_drop_idx = set()
if _valid.sum() > 1:
    if _valid.sum() <= PHASH_MAX_N_FOR_PAIRWISE:
        hash_bits = np.array([h.hash.flatten() for h in df.loc[_valid, 'phash']]).astype(np.int16)
        positions = df.index[_valid].to_numpy()
        ones = hash_bits.sum(1)
        dot = hash_bits @ hash_bits.T
        hamming = ones[:, None] + ones[None, :] - 2 * dot          # O(n²) time, O(n²) small-int memory
        np.fill_diagonal(hamming, hash_bits.shape[1] + 1)

        # Union-find so transitively-linked near-dupes group together
        parent = {int(p): int(p) for p in positions}
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra

        ii, jj = np.where(hamming <= PHASH_HAMMING_THRESHOLD)
        for a, b in zip(ii, jj):
            if a < b:
                union(int(positions[a]), int(positions[b]))

        groups = {}
        for p in positions:
            groups.setdefault(find(int(p)), []).append(int(p))
        dup_groups = {k: v for k, v in groups.items() if len(v) > 1}

        print(f"\nNear-duplicate (pHash, Hamming ≤ {PHASH_HAMMING_THRESHOLD}) groups found: {len(dup_groups)}")
        for root, members in list(dup_groups.items())[:10]:
            print(f"   group of {len(members)}: {[df.loc[m, 'path'] for m in members]}")
            near_dup_drop_idx.update(members[1:])  # keep first, drop the rest
        if len(dup_groups) > 10:
            print(f"   ... and {len(dup_groups) - 10} more groups")
    else:
        print(f"⚠️  {_valid.sum()} images — skipping O(n²) pairwise pHash comparison "
              f"(> {PHASH_MAX_N_FOR_PAIRWISE}). Consider LSH/bucketing for larger datasets.")

if near_dup_drop_idx:
    initial_len = len(df)
    df = df.drop(index=list(near_dup_drop_idx)).reset_index(drop=True)
    print(f"Removed {initial_len - len(df)} near-duplicate images. New dataset size: {len(df)}")
else:
    print("✅ No near-duplicate images detected (pHash).")

df = df.drop(columns=['file_hash', 'phash'])

# ── 5d: Encode labels ────────────────────────────────────────────────────────
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
CFG.CLASS_NAMES = list(le.classes_)
CFG.NUM_CLASSES = len(CFG.CLASS_NAMES)

# ── 5e: Patient/subject-aware split (Reviewer #5 — patient-level leakage) ──
# `train_test_split` on a flat image list can put two scans of the SAME
# patient into both train and test, letting the model memorise
# patient-specific texture instead of learning tumour features — this alone
# can produce a suspiciously high AUC. We use GroupShuffleSplit so that all
# images belonging to one inferred "group" end up in exactly one split.
#
# ⚠️ KNOWN LIMITATION: Br35H (Kaggle 'ahmedhamada0/brain-tumor-detection')
# ships flat, anonymised filenames (e.g. 'Y12.jpg', 'no34.jpg') with NO
# genuine patient/subject metadata. `infer_group_id` below is a best-effort
# heuristic that only groups together exact re-saves / obvious augmented
# copies of the same base filename — it CANNOT recover true patient
# identity that was never recorded. If every inferred group turns out to be
# a singleton (checked below), this split behaves like a plain per-image
# split and the possibility of patient-level leakage from the ORIGINAL data
# collection cannot be ruled out. Report this explicitly as a dataset
# limitation in any write-up; do not claim patient-level leakage is
# "handled" on Br35H specifically.
def infer_group_id(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    stem = re.sub(r'(_aug\d+|\(\d+\)|_copy\d*|-copy\d*)$', '', stem, flags=re.IGNORECASE)
    return stem

df['group_id'] = df['path'].apply(infer_group_id)
if CFG.SCAN_GROUP_SPLIT:
    # Br35H contains exact, rescaled and LEFT-RIGHT MIRRORED copies of many slices; file-name
    # stems do not capture them. Group by scan cluster instead (br35h_data.scan_clusters:
    # MD5 / pHash<=5 / mirrored pHash + pixel correlation). ~631 clusters expected here.
    import br35h_data as bd
    df = df.reset_index(drop=True)
    df['group_id'] = [f"scan{c}" for c in bd.scan_clusters(df['path'].tolist())]
    print(f"group_id = scan cluster: {df['group_id'].nunique()} groups for {len(df)} images")
n_groups, n_rows = df['group_id'].nunique(), len(df)
if n_groups == n_rows:
    print(f"\n⚠️  DATA-AUDIT WARNING: all {n_rows} images resolved to unique inferred group "
          f"ids. Br35H ships no real patient/subject metadata, so GroupShuffleSplit below "
          f"is running with ~1 image per group and CANNOT rule out patient-level leakage "
          f"in the original data collection. Report this as a known dataset limitation.")
else:
    print(f"\nℹ️  Inferred {n_groups} groups from {n_rows} images "
          f"({n_rows - n_groups} images grouped with a same-stem sibling).")

gss1 = GroupShuffleSplit(n_splits=1, test_size=CFG.TEST_RATIO, random_state=CFG.SEED)
tv_idx, test_idx = next(gss1.split(df, df['label_enc'], groups=df['group_id']))
train_val = df.iloc[tv_idx].reset_index(drop=True)
test_df   = df.iloc[test_idx].reset_index(drop=True)

val_size = CFG.VAL_RATIO / (CFG.TRAIN_RATIO + CFG.VAL_RATIO)
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=CFG.SEED)
tr_idx, va_idx = next(gss2.split(train_val, train_val['label_enc'], groups=train_val['group_id']))
train_df = train_val.iloc[tr_idx].reset_index(drop=True)
val_df   = train_val.iloc[va_idx].reset_index(drop=True)

# Hard assertions: NO group id may appear in more than one split. If this
# fires, something upstream (e.g. a manual DATASET_PATH edit) broke the
# split contract and training must not proceed silently.
assert set(train_df['group_id']) & set(val_df['group_id']) == set(), "Group leakage: train/val!"
assert set(train_val['group_id']) & set(test_df['group_id']) == set(), "Group leakage: train_val/test!"
assert set(train_df['group_id']) & set(test_df['group_id']) == set(), "Group leakage: train/test!"

print(f"\n✅ Split complete (patient/subject-grouped):")
print(f"   Train : {len(train_df):>5}  ({len(train_df)/len(df)*100:.1f}%)")
print(f"   Val   : {len(val_df):>5}  ({len(val_df)/len(df)*100:.1f}%)")
print(f"   Test  : {len(test_df):>5}  ({len(test_df)/len(df)*100:.1f}%)")
print(f"   Classes (encoded order) : {CFG.CLASS_NAMES}")
print(f"   Train class balance : {train_df['label_enc'].value_counts(normalize=True).round(3).to_dict()}")

# ── 5f: Augmentations — tuned for grayscale brain MRI, not natural photos ────
# (dropped HueSaturationValue: meaningless on grayscale-as-RGB scans;
#  dropped VerticalFlip: anatomically implausible for axial brain MRI)
train_tfm = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12, rotate_limit=15, p=0.6),
    A.OneOf([
        A.CLAHE(clip_limit=4.0),
        A.RandomBrightnessContrast(0.25, 0.25),
        A.RandomGamma(gamma_limit=(80, 120)),
    ], p=0.6),
    A.GaussNoise(var_limit=(5, 30), p=0.25),
    A.CoarseDropout(max_holes=6, max_height=CFG.IMG_SIZE // 12,
                     max_width=CFG.IMG_SIZE // 12, fill_value=0, p=0.25),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_tfm = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# ── 5g: Dataset class — skips an unreadable file instead of crashing mid-epoch ──
class MedDataset(Dataset):
    def __init__(self, df, tfm=None, return_path=False):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.return_path = return_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = np.array(Image.open(row['path']).convert('RGB'))
        except Exception as e:
            print(f"⚠️  Skipping unreadable image: {row['path']} ({e})")
            return self.__getitem__((idx + 1) % len(self.df))
        lbl = int(row['label_enc'])
        if self.tfm:
            img = self.tfm(image=img)['image']
        if self.return_path:
            return img, lbl, row['path']
        return img, lbl

# ── 5h: Class weights for the loss function ─────────────────────────────────
# Reviewer #6: WeightedRandomSampler is unnecessary here — Br35H is close to
# balanced, and resampling adds nondeterminism (each epoch sees a different
# resampled subset) without a corresponding benefit. Class imbalance (such
# as it is) is handled directly inside the loss (CombLoss -> SmoothCE/Focal),
# which is simpler, deterministic given a fixed seed, and doesn't distort
# the empirical class distribution the model is validated/tested against.
def compute_class_weights(df):
    counts = df['label_enc'].value_counts().sort_index().values
    w = 1.0 / counts
    w = w / w.sum()
    return torch.FloatTensor(w)

class_weights = compute_class_weights(train_df).to(DEVICE)

train_ds = MedDataset(train_df, train_tfm)
val_ds   = MedDataset(val_df,   val_tfm)
test_ds  = MedDataset(test_df,  val_tfm, return_path=True)

NUM_WORKERS = 2  # set to 0 if you hit DataLoader worker crashes on Colab
train_loader = DataLoader(train_ds, CFG.BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   CFG.BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  CFG.BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

print(f"\n✅ DataLoaders ready: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)} batches")

In [ ]:
from collections import defaultdict

def tensor_to_numpy(tensor):
    """Converts a PyTorch tensor to a NumPy array for plotting."""
    # Assumes CxHxW and converts to HxWxC
    return tensor.cpu().numpy().transpose(1, 2, 0)

def denorm(t):
    """Denormalizes an image tensor back to displayable pixel values."""
    img = t.detach().permute(1,2,0).cpu().numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1).astype(np.float32)

# ── 6.1  Sample images — one row per class ───────────────────────────────────
SAMPLES_PER_CLASS = 5

class_samples = defaultdict(list)  # label → list of tensors per class index
for img, label, _ in test_ds: # Changed to unpack 3 values from test_ds
    if len(class_samples[label]) < SAMPLES_PER_CLASS:
        class_samples[label].append(img)
    # ✅ FIX: guard must be OUTSIDE the generator expression.
    # The old code had `if len(class_samples) == NUM_CLASSES` as a generator
    # filter, so all([]) → True caused an immediate break after image #1.
    if (len(class_samples) == CFG.NUM_CLASSES and
            all(len(v) == SAMPLES_PER_CLASS for v in class_samples.values())):
        break

fig, axes = plt.subplots(CFG.NUM_CLASSES, SAMPLES_PER_CLASS,
                          figsize=(SAMPLES_PER_CLASS * 3, CFG.NUM_CLASSES * 3))
fig.suptitle('Sample Images per Class', fontsize=16, fontweight='bold', y=1.01)

for row, cls_idx in enumerate(range(CFG.NUM_CLASSES)):
    samples = class_samples.get(cls_idx, [])
    for col in range(SAMPLES_PER_CLASS):
        ax = axes[row][col]
        ax.axis('off')                          # ✅ FIX: always hide axes first
        if col < len(samples):                  #    so empty slots stay clean
            # Apply denormalization before displaying
            ax.imshow(denorm(samples[col]))
            ax.set_title(CFG.CLASS_NAMES[cls_idx], fontsize=10)
        if col == 0:
            ax.set_ylabel(CFG.CLASS_NAMES[cls_idx], fontsize=11,
                          fontweight='bold', rotation=0,
                          labelpad=80, va='center')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅  Sample images plotted.")

## Cell 7 — Hybrid CNN–ViT Model Architecture

In [ ]:
# ── CBAM Attention ────────────────────────────────────────────────────────
class CBAM(nn.Module):
    def __init__(self, ch, r=16, k=7):
        super().__init__()
        r = max(1, min(r, ch))  # guard against r > ch on small backbones
        self.ch_avg = nn.AdaptiveAvgPool2d(1)
        self.ch_max = nn.AdaptiveMaxPool2d(1)
        self.ch_mlp = nn.Sequential(nn.Flatten(),
                                     nn.Linear(ch, ch // r, bias=False), nn.ReLU(),
                                     nn.Linear(ch // r, ch, bias=False))
        self.sp_conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)
        self.sig = nn.Sigmoid()

    def forward(self, x):
        ch = self.sig((self.ch_mlp(self.ch_avg(x)) + self.ch_mlp(self.ch_max(x))).unsqueeze(2).unsqueeze(3))
        x = x * ch
        sp = self.sig(self.sp_conv(torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)))
        return x * sp

# ── CNN Branch: EfficientNet + CBAM ─────────────────────────────────────────
class CNNBranch(nn.Module):
    def __init__(self, name=CFG.CNN_BACKBONE, out=CFG.FUSION_DIM):
        super().__init__()
        self.backbone = timm.create_model(name, pretrained=True, num_classes=0, global_pool='')
        with torch.no_grad():
            ch = self.backbone.forward_features(torch.zeros(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)).shape[1]
        self.cbam = CBAM(ch)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Flatten(), nn.LayerNorm(ch),
                                   nn.Linear(ch, out), nn.GELU(), nn.Dropout(CFG.DROPOUT))

    def forward(self, x):
        f = self.backbone.forward_features(x)
        f = self.cbam(f)
        return self.proj(self.pool(f))

    def last_conv_layer(self):
        """Best-effort lookup of the final conv layer, used as the GradCAM target.
        Works across different timm EfficientNet variants instead of hardcoding
        an attribute name (e.g. 'conv_head') that may not exist in every version."""
        for attr in ('conv_head', 'conv_head_2', 'conv_final'):
            if hasattr(self.backbone, attr):
                return getattr(self.backbone, attr)
        convs = [m for m in self.backbone.modules() if isinstance(m, nn.Conv2d)]
        if not convs:
            raise RuntimeError("Could not find a Conv2d layer in the CNN backbone for GradCAM.")
        return convs[-1]

# ── ViT Branch ───────────────────────────────────────────────────────────────
class ViTBranch(nn.Module):
    def __init__(self, name=CFG.VIT_BACKBONE, out=CFG.FUSION_DIM):
        super().__init__()
        self.vit = timm.create_model(name, pretrained=True, num_classes=0)
        vit_dim = self.vit.embed_dim
        self.proj = nn.Sequential(nn.LayerNorm(vit_dim),
                                   nn.Linear(vit_dim, out), nn.GELU(), nn.Dropout(CFG.DROPOUT))

    def forward(self, x):
        return self.proj(self.vit(x))

# ── Cross-Attention Fusion ────────────────────────────────────────────────────
class CrossAttnFusion(nn.Module):
    def __init__(self, dim=CFG.FUSION_DIM, heads=8):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.n2 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=0.1, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(),
                                  nn.Dropout(0.1), nn.Linear(dim * 4, dim))
        self.n3 = nn.LayerNorm(dim)

    def forward(self, cnn, vit):
        q = cnn.unsqueeze(1); k = vit.unsqueeze(1)
        out, w = self.attn(self.n1(q), self.n2(k), self.n2(k))
        out = q + out
        out = out + self.ffn(self.n3(out))
        return out.squeeze(1), w

# ── Main Hybrid Model ─────────────────────────────────────────────────────────
class HybridModel(nn.Module):
    """Hybrid CNN-ViT for binary brain-tumor classification:
    EfficientNet + CBAM  -->  Cross-Attention + Learnable Gating  -->  Classifier
    ViT                   -/
    """
    def __init__(self, nc=CFG.NUM_CLASSES, dim=CFG.FUSION_DIM):
        super().__init__()
        self.cnn = CNNBranch(out=dim)
        self.vit = ViTBranch(out=dim)
        self.fusion = CrossAttnFusion(dim)
        self.gate = nn.Sequential(nn.Linear(dim * 2, 2), nn.Softmax(dim=-1))
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim // 2),
                                   nn.GELU(), nn.Dropout(CFG.DROPOUT), nn.Linear(dim // 2, nc))
        self.cnn_aux = nn.Linear(dim, nc)
        self.vit_aux = nn.Linear(dim, nc)

    def forward(self, x, return_feats=False):
        cf = self.cnn(x); vf = self.vit(x)
        fused, attn = self.fusion(cf, vf)
        g = self.gate(torch.cat([cf, vf], dim=-1))
        z = g[:, 0:1] * cf + g[:, 1:2] * vf + fused
        logits = self.head(z)
        if return_feats:
            return logits, z, attn
        return logits, self.cnn_aux(cf), self.vit_aux(vf)

model = HybridModel().to(DEVICE)
tp = sum(p.numel() for p in model.parameters())
tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Hybrid Model initialized!")
print(f"   Total parameters     : {tp:,}")
print(f"   Trainable parameters : {tr:,}")
print(f"   CNN : {CFG.CNN_BACKBONE} + CBAM")
print(f"   ViT : {CFG.VIT_BACKBONE}")
print(f"   Fusion: Cross-Attention + Learnable Gating")
free_mem()

## Cell 8 — Loss, Optimizer, Scheduler, EMA.

## Reliability adjustments applied

Changes in this version:
- Added duplicate image leakage check.
- Reduced label smoothing to avoid under-confident predictions.
- Increased dropout for better generalization.
- Increased epochs with early stopping retained.
- Added precision and recall reporting.


In [ ]:
# ── Label Smoothing + Class Weights ──────────────────────────────────────────
class SmoothCE(nn.Module):
    def __init__(self, nc, sm=0.1, w=None):
        super().__init__()
        self.nc = nc; self.sm = sm; self.w = w
    def forward(self, pred, tgt):
        conf = 1 - self.sm; sv = self.sm / max(1, self.nc - 1)
        oh = torch.zeros_like(pred).scatter_(1, tgt.unsqueeze(1), 1)
        st = oh * conf + (1 - oh) * sv
        lp = F.log_softmax(pred, dim=-1)
        loss = -(st * lp).sum(-1)
        if self.w is not None:
            loss = loss * self.w[tgt]
        return loss.mean()

class FocalLoss(nn.Module):
    def __init__(self, a=0.25, g=2.0, w=None):
        super().__init__()
        self.a = a; self.g = g; self.w = w
    def forward(self, inp, tgt):
        ce = F.cross_entropy(inp, tgt, weight=self.w, reduction='none')
        pt = torch.exp(-ce)
        return (self.a * (1 - pt) ** self.g * ce).mean()

class CombLoss(nn.Module):
    def __init__(self, nc, w=None, aux=0.3):
        super().__init__()
        self.main = SmoothCE(nc, CFG.LABEL_SMOOTH, w)
        self.focal = FocalLoss(w=w)
        self.aux = aux
    def forward(self, ml, cl, vl, tgt):
        lm = 0.5 * self.main(ml, tgt) + 0.5 * self.focal(ml, tgt)
        la = (self.main(cl, tgt) + self.main(vl, tgt)) / 2 if cl is not None else 0
        return lm + self.aux * la

criterion = CombLoss(CFG.NUM_CLASSES, class_weights)

# ── AdamW with differential LR ────────────────────────────────────────────────
def build_optimizer(m, base_lr=CFG.LR):
    return optim.AdamW([
        {'params': m.cnn.parameters(), 'lr': base_lr * 0.1},
        {'params': m.vit.parameters(), 'lr': base_lr * 0.1},
        {'params': list(m.fusion.parameters()) + list(m.gate.parameters()) +
                    list(m.head.parameters()) + list(m.cnn_aux.parameters()) +
                    list(m.vit_aux.parameters()), 'lr': base_lr},
    ], weight_decay=CFG.WEIGHT_DECAY)

optimizer = build_optimizer(model)

# ── Warmup + cosine scheduler in plain PyTorch (replaces the version-fragile
#    timm.scheduler.CosineLRScheduler, whose .step(epoch) API changed across
#    timm releases and was a likely crash source) ──────────────────────────
def build_scheduler(opt, total_epochs, warmup_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        prog = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * min(prog, 1.0))) * 0.99 + 0.01
    return optim.lr_scheduler.LambdaLR(opt, lr_lambda)

scheduler = build_scheduler(optimizer, CFG.NUM_EPOCHS, CFG.WARMUP_EPOCHS)

# ── Lightweight EMA (replaces the torch-ema package to avoid its API drift) ───
class SimpleEMA:
    def __init__(self, params, decay=0.999):
        self.decay = decay
        self.shadow = [p.detach().clone() for p in params]
        self.collected = None
    def update(self, params):
        with torch.no_grad():
            for s, p in zip(self.shadow, params):
                s.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    def apply_to(self, params):
        self.collected = [p.detach().clone() for p in params]
        with torch.no_grad():
            for p, s in zip(params, self.shadow):
                p.copy_(s)
    def restore(self, params):
        if self.collected is None:
            return
        with torch.no_grad():
            for p, c in zip(params, self.collected):
                p.copy_(c)
        self.collected = None

ema = SimpleEMA(list(model.parameters()), decay=0.999)
scaler = make_scaler()

# ── MixUp / CutMix ───────────────────────────────────────────────────────────
def mixup(x, y, a=0.2):
    lam = np.random.beta(a, a)
    idx = torch.randperm(x.size(0), device=DEVICE)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def rand_bbox(sz, lam):
    W, H = sz[2], sz[3]; cr = np.sqrt(1 - lam)
    cw, ch = int(W * cr), int(H * cr)
    cx, cy = np.random.randint(W), np.random.randint(H)
    return (np.clip(cx - cw // 2, 0, W), np.clip(cy - ch // 2, 0, H),
            np.clip(cx + cw // 2, 0, W), np.clip(cy + ch // 2, 0, H))

def cutmix(x, y, a=1.0):
    lam = np.random.beta(a, a)
    idx = torch.randperm(x.size(0), device=DEVICE)
    x1, y1, x2, y2 = rand_bbox(x.size(), lam)
    x = x.clone(); x[:, :, x1:x2, y1:y2] = x[idx, :, x1:x2, y1:y2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (x.size(2) * x.size(3)))
    return x, y, y[idx], lam

print("✅ Loss, optimizer, scheduler, EMA, MixUp/CutMix ready!")

## Cell 9 — Training Loop

In [ ]:
history = {k: [] for k in ['tr_loss', 'va_loss', 'tr_acc', 'va_acc', 'va_auc', 'va_f1', 'lr']}

def train_epoch(mdl, ldr, opt, crit, sc, ep):
    mdl.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, lbls in tqdm(ldr, desc=f'Ep {ep} [Train]', leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        use_mx = CFG.USE_MIXUP and np.random.rand() < 0.5
        use_cx = CFG.USE_CUTMIX and np.random.rand() < 0.5 and not use_mx
        if use_mx:
            imgs, ya, yb, lam = mixup(imgs, lbls, CFG.MIXUP_ALPHA)
        elif use_cx:
            imgs, ya, yb, lam = cutmix(imgs, lbls)
        opt.zero_grad(set_to_none=True)
        try:
            with autocast_ctx():
                ml, cl, vl = mdl(imgs)
                if use_mx or use_cx:
                    loss = lam * crit(ml, cl, vl, ya) + (1 - lam) * crit(ml, cl, vl, yb)
                else:
                    loss = crit(ml, cl, vl, lbls)
            sc.scale(loss).backward()
            sc.unscale_(opt)
            nn.utils.clip_grad_norm_(mdl.parameters(), CFG.GRAD_CLIP)
            sc.step(opt); sc.update()
            ema.update(list(mdl.parameters()))
        except torch.cuda.OutOfMemoryError:
            print(f"⚠️  OOM on a batch — skipping it and clearing cache. "
                  f"If this repeats, lower CFG.BATCH_SIZE or check LOW_MEMORY_MODE.")
            opt.zero_grad(set_to_none=True)
            free_mem()
            continue
        correct += (ml.argmax(1) == lbls).sum().item()
        total += lbls.size(0); loss_sum += loss.item()
    return loss_sum / max(1, len(ldr)), correct / max(1, total)

@torch.no_grad()
def validate(mdl, ldr, crit):
    mdl.eval()
    ls, preds, labels, probs = 0.0, [], [], []
    for batch in ldr:
        imgs, lbls = batch[0], batch[1]
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        with autocast_ctx():
            ml, cl, vl = mdl(imgs)
            ls += crit(ml, cl, vl, lbls).item()
        p = F.softmax(ml, 1)
        preds.extend(p.argmax(1).cpu().numpy())
        labels.extend(lbls.cpu().numpy())
        probs.extend(p.float().cpu().numpy())
    yt, yp, ypr = np.array(labels), np.array(preds), np.array(probs)
    acc = (yp == yt).mean()
    f1 = f1_score(yt, yp, average='weighted')
    try:
        auc = roc_auc_score(yt, ypr[:, 1] if CFG.NUM_CLASSES == 2 else ypr,
                             multi_class='ovr' if CFG.NUM_CLASSES > 2 else 'raise',
                             average='macro' if CFG.NUM_CLASSES > 2 else None)
    except ValueError:
        auc = float('nan')  # e.g. a split landed with only one class present
    return ls / max(1, len(ldr)), acc, f1, auc, yt, yp, ypr

best_auc, patience, PATIENCE = 0.0, 0, 8

for epoch in range(1, CFG.NUM_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, scaler, epoch)
    va_loss, va_acc, va_f1, va_auc, *_ = validate(model, val_loader, criterion)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    for k, v in zip(['tr_loss', 'va_loss', 'tr_acc', 'va_acc', 'va_auc', 'va_f1', 'lr'],
                     [tr_loss, va_loss, tr_acc, va_acc, va_auc, va_f1, lr]):
        history[k].append(v)

    print(f"[{epoch:02d}/{CFG.NUM_EPOCHS}] "
          f"Loss {tr_loss:.4f}/{va_loss:.4f} | Acc {tr_acc:.4f}/{va_acc:.4f} | "
          f"AUC {va_auc:.4f} | F1 {va_f1:.4f} | LR {lr:.2e} | {time.time()-t0:.1f}s")

    if va_auc > best_auc:
        best_auc = va_auc; patience = 0
        ema.apply_to(list(model.parameters()))
        torch.save({'epoch': epoch, 'state': model.state_dict(), 'auc': best_auc},
                    f'{CFG.CHECKPOINT}best_model.pth')
        ema.restore(list(model.parameters()))
        print(f"   ✅ Saved! Best AUC = {best_auc:.4f}")
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f"⚠️  Early stopping at epoch {epoch}")
            break
    free_mem()

print(f"\n🏁 Training complete! Best AUC = {best_auc:.4f}")

## Cell 10 — Evaluation on Test Set + Training Curves

In [ ]:
ckpt = torch.load(f'{CFG.CHECKPOINT}best_model.pth', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['state'])
print(f"✅ Loaded checkpoint — Epoch {ckpt['epoch']} | Val AUC {ckpt['auc']:.4f}")

te_loss, te_acc, te_f1, te_auc, y_true, y_pred, y_prob = validate(model, test_loader, criterion)

print(f"\n{'='*52}")
print(f"  TEST RESULTS — Br35H Brain Tumor Classification")
print(f"{'='*52}")
print(f"  Accuracy  : {te_acc:.4f} ({te_acc*100:.2f}%)")
print(f"  AUC-ROC   : {te_auc:.4f}")
print(f"  F1-Score  : {te_f1:.4f}")
print(f"  MCC       : {matthews_corrcoef(y_true, y_pred):.4f}")
print(f"  Kappa     : {cohen_kappa_score(y_true, y_pred):.4f}")
print(f"{'='*52}")
print(f"\n{classification_report(y_true, y_pred, target_names=CFG.CLASS_NAMES, digits=4)}")

ep = range(1, len(history['tr_loss']) + 1)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training History & Test Evaluation — Br35H', fontsize=15, fontweight='bold')

axes[0,0].plot(ep, history['tr_loss'], '#e74c3c', lw=2, label='Train')
axes[0,0].plot(ep, history['va_loss'], '#3498db', lw=2, label='Val')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ep, history['tr_acc'], '#e74c3c', lw=2, label='Train')
axes[0,1].plot(ep, history['va_acc'], '#3498db', lw=2, label='Val')
axes[0,1].set_title('Accuracy'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

axes[0,2].plot(ep, history['va_auc'], '#2ecc71', lw=2, label='AUC')
axes[0,2].plot(ep, history['va_f1'],  '#9b59b6', lw=2, label='F1')
axes[0,2].set_title('Val AUC & F1'); axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

cm = confusion_matrix(y_true, y_pred, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1,0],
            xticklabels=CFG.CLASS_NAMES, yticklabels=CFG.CLASS_NAMES)
axes[1,0].set_title('Confusion Matrix (Normalised)')

fpr, tpr, _ = roc_curve(y_true, y_prob[:,1])
axes[1,1].plot(fpr, tpr, '#e74c3c', lw=2, label=f'AUC={te_auc:.4f}')
axes[1,1].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[1,1].plot([0,1],[0,1],'k--'); axes[1,1].set_title('ROC Curve')
axes[1,1].set_xlabel('FPR'); axes[1,1].set_ylabel('TPR')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

axes[1,2].plot(ep, history['lr'], '#f39c12', lw=2)
axes[1,2].set_title('Learning Rate'); axes[1,2].set_yscale('log'); axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.RESULTS}training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curves saved!")
free_mem()

In [ ]:
ep = range(1, len(history['lr']) + 1)
plt.figure(figsize=(10, 6))
plt.plot(ep, history['lr'], label='Learning Rate', color='purple')
plt.title('Learning Rate Schedule Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
class ModelWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.model = m
    def forward(self, x):
        logits, _, _ = self.model(x)
        return logits

wrapped = ModelWrapper(model)
model.eval()

# Collect all samples from the test_loader
all_sample_imgs = []
all_sample_lbls = []
all_sample_paths = []

for imgs, lbls, paths in test_loader:
    all_sample_imgs.append(imgs)
    all_sample_lbls.append(lbls)
    all_sample_paths.extend(paths)

sample_imgs = torch.cat(all_sample_imgs)
sample_lbls = torch.cat(all_sample_lbls)
sample_paths = all_sample_paths

def denorm(t):
    """Denormalizes an image tensor back to displayable pixel values."""
    img = t.detach().permute(1,2,0).cpu().numpy() # Added .cpu() here to fix the RuntimeError
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1).astype(np.float32)

# ── A: GradCAM / GradCAM++ / ScoreCAM ────────────────
def run_gradcam(n=4, method='gradcam++'):
    target_layer = [model.cnn.last_conv_layer()]
    CAM_MAP = {'gradcam': GradCAM, 'gradcam++': GradCAMPlusPlus,
               'scorecam': ScoreCAM, 'layercam': LayerCAM}
    CAM = CAM_MAP.get(method, GradCAMPlusPlus)
    n = min(n, len(sample_imgs))

    fig, axes = plt.subplots(3, n, figsize=(n*3.5, 10))
    fig.suptitle(f'{method.upper()} Explanations — Br35H', fontsize=13, fontweight='bold')
    if n == 1:
        axes = axes.reshape(3, 1)

    with CAM(model=wrapped, target_layers=target_layer) as cam:
        for i in range(n):
            it  = sample_imgs[i:i+1].to(DEVICE)
            lbl = sample_lbls[i].item()
            gc  = cam(it, targets=[ClassifierOutputTarget(lbl)])[0]
            raw = denorm(sample_imgs[i])
            ov  = show_cam_on_image(raw, gc, use_rgb=True)
            axes[0,i].imshow(raw);            axes[0,i].set_title(CFG.CLASS_NAMES[lbl]); axes[0,i].axis('off')
            axes[1,i].imshow(gc, cmap='jet'); axes[1,i].set_title('Heatmap');             axes[1,i].axis('off')
            axes[2,i].imshow(ov);             axes[2,i].set_title('Overlay');              axes[2,i].axis('off')

    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}{method}_explanations.png', dpi=150, bbox_inches='tight')
    plt.show(); print(f"✅ {method.upper()} saved!")

try:
    run_gradcam(method='gradcam++')
except Exception as e:
    print(f"☢️  GradCAM++ failed: {e}")

try:
    run_gradcam(method='scorecam')   # slow — one forward pass per channel
except Exception as e:
    print(f"☢️  ScoreCAM failed: {e}")

# ── B: ViT Attention Rollout ────────────────────────────
def _ensure_eager_attention():
    # timm's fused (PyTorch SDPA) attention path never materializes the
    # attention-probability matrix, so there'd be nothing meaningful to hook.
    # Forcing the eager softmax path lets us read real attention weights.
    for blk in model.vit.vit.blocks:
        if hasattr(blk.attn, 'fused_attn'):
            blk.attn.fused_attn = False

_ensure_eager_attention()

def attn_rollout(img_t, discard=0.9):
    hooks, atts = [], []
    for blk in model.vit.vit.blocks:
        hooks.append(blk.attn.attn_drop.register_forward_hook(
            lambda m, i, o: atts.append(o.detach().cpu())))
    model.eval()
    with torch.no_grad():
        wrapped(img_t.to(DEVICE))
    for h in hooks:
        h.remove()
    if not atts:
        raise RuntimeError("No attention maps captured.")
    R = torch.eye(atts[0].size(-1))
    for a in atts:
        a = a.mean(1)
        a = a + torch.eye(a.size(-1))
        a = a / a.sum(-1, keepdim=True)
        fl = a.view(a.size(0), -1)
        th = fl.quantile(discard, -1, keepdim=True)
        a = (a > th.unsqueeze(-1)).float() * a
        a = a / (a.sum(-1, keepdim=True) + 1e-6)
        R = torch.matmul(a[0], R)
    mask = R[0, 1:]
    n = int(np.sqrt(mask.shape[0]))
    mask = mask[:n*n].reshape(n, n)
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    return mask.numpy()

def vis_rollout(n=4):
    n = min(n, len(sample_imgs))
    fig, axes = plt.subplots(2, n, figsize=(n*4, 8))
    fig.suptitle('ViT Attention Rollout — Br35H', fontsize=13, fontweight='bold')
    if n == 1:
        axes = axes.reshape(2, 1)
    for i in range(n):
        mask = attn_rollout(sample_imgs[i:i+1])
        msk  = np.array(Image.fromarray(mask).resize((CFG.IMG_SIZE, CFG.IMG_SIZE), Image.LANCZOS))
        raw  = denorm(sample_imgs[i])
        lbl  = sample_lbls[i].item()
        axes[0,i].imshow(raw); axes[0,i].set_title(CFG.CLASS_NAMES[lbl]); axes[0,i].axis('off')
        axes[1,i].imshow(raw); axes[1,i].imshow(msk, cmap='jet', alpha=0.5)
        axes[1,i].set_title('Attn Rollout'); axes[1,i].axis('off')
    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}attention_rollout.png', dpi=150, bbox_inches='tight')
    plt.show(); print("✅ Attention Rollout saved!")

try:
    vis_rollout()
except Exception as e:
    print(f"☢️  Attention rollout failed: {e}")

# # ── C: Integrated Gradients (SmoothGrad²) ───────────────────
# def vis_ig(n=4, nt_samples_val=8):
#     n = min(n, len(sample_imgs))
#     ig = IntegratedGradients(wrapped)
#     fig, axes = plt.subplots(3, n, figsize=(n*4, 12))
#     fig.suptitle('Integrated Gradients (SmoothGrad²) — Br35H', fontsize=13, fontweight='bold')
#     if n == 1:
#         axes = axes.reshape(3, 1)
#     for i in range(n):
#         it  = sample_imgs[i:i+1].to(DEVICE).requires_grad_(True)
#         lbl = sample_lbls[i].item()
#         nt  = NoiseTunnel(ig, nt_type='smoothgrad_sq', nt_samples=nt_samples_val, stdevs=0.02)
#         att = nt.attribute(it, baselines=torch.zeros_like(it), target=lbl)
#         am  = np.abs(att[0].permute(1,2,0).cpu().detach().numpy()).mean(-1)
#         am  = (am - am.min()) / (am.max() - am.min() + 1e-8)
#         raw = denorm(sample_imgs[i])
#         axes[0,i].imshow(raw);            axes[0,i].set_title(CFG.CLASS_NAMES[lbl]); axes[0,i].axis('off')
#         axes[1,i].imshow(am, cmap='hot'); axes[1,i].set_title('Attribution');         axes[1,i].axis('off')
#         axes[2,i].imshow(raw); axes[2,i].imshow(am, cmap='hot', alpha=0.5)
#         axes[2,i].set_title('Overlay'); axes[2,i].axis('off')
#     plt.tight_layout()
#     plt.savefig(f'{CFG.RESULTS}integrated_gradients.png', dpi=150, bbox_inches='tight')
#     plt.show(); print("✅ Integrated Gradients saved!")

# try:
#     print("Attempting Integrated Gradients with nt_samples=4 (memory-efficient)...")
#     vis_ig(nt_samples_val=4)
# except Exception as e:
#     print(f"☢️  Integrated Gradients failed: {e}")

free_mem()

## Cell 12 — LIME & SHAP Explanations

In [ ]:
# ── D: LIME ───────────────────────────────────────────────────────────────
def vis_lime(n=3, num_samples=150):
    n = min(n, len(sample_imgs))
    explainer = lime_image.LimeImageExplainer(random_state=CFG.SEED)

    def predict_fn(images, chunk=32):
        # Batched in chunks instead of pushing all ~150+ perturbed images through
        # both backbones in one GPU call — that single-shot version is a common
        # OOM source on a free Colab GPU.
        model.eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(images), chunk):
                batch = torch.stack([val_tfm(image=img)['image']
                                      for img in images[i:i+chunk]]).to(DEVICE)
                logits, _, _ = model(batch)
                out.append(F.softmax(logits, 1).cpu().numpy())
        return np.concatenate(out, axis=0)

    fig, axes = plt.subplots(2, n, figsize=(n*5, 10))
    fig.suptitle('LIME — Superpixel Explanations — Br35H', fontsize=13, fontweight='bold')
    if n == 1:
        axes = axes.reshape(2, 1)

    for i in range(n):
        raw    = denorm(sample_imgs[i])
        raw_u8 = (raw * 255).astype(np.uint8)
        lbl    = sample_lbls[i].item()
        exp = explainer.explain_instance(
            raw_u8, predict_fn, top_labels=CFG.NUM_CLASSES, hide_color=0,
            num_samples=num_samples,
            segmentation_fn=SegmentationAlgorithm('slic', n_segments=60))
        temp, mask = exp.get_image_and_mask(lbl, positive_only=True, num_features=8, hide_rest=False)
        axes[0,i].imshow(raw);  axes[0,i].set_title(CFG.CLASS_NAMES[lbl]); axes[0,i].axis('off')
        axes[1,i].imshow(temp); axes[1,i].set_title('LIME Regions');        axes[1,i].axis('off')

    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}lime_explanations.png', dpi=150, bbox_inches='tight')
    plt.show(); print("✅ LIME saved!")

try:
    vis_lime()
except Exception as e:
    print(f"⚠️  LIME failed: {e}")

free_mem()

# ── E: SHAP — GradientExplainer (far more robust than DeepExplainer for
#      GELU/SiLU-based hybrid CNN-ViT models, which DeepExplainer's hook-based
#      activation rules generally don't support well) ───────────────────────
def vis_shap(n=3, bg=12, nsamples=50):
    model.eval()
    n = min(n, len(sample_imgs))
    bg_imgs = next(iter(train_loader))[0][:bg].to(DEVICE)
    explainer = shap.GradientExplainer(wrapped, bg_imgs)
    test_t = sample_imgs[:n].to(DEVICE)
    shap_values, indexes = explainer.shap_values(test_t, nsamples=nsamples, ranked_outputs=CFG.NUM_CLASSES)
    if isinstance(shap_values, list):
        svs = shap_values
    else:
        svs = [shap_values[..., c] for c in range(shap_values.shape[-1])]

    fig, axes = plt.subplots(n, CFG.NUM_CLASSES + 1, figsize=((CFG.NUM_CLASSES+1)*4, n*4))
    fig.suptitle('SHAP GradientExplainer — Per-Class Attribution — Br35H', fontsize=13, fontweight='bold')
    if n == 1:
        axes = axes.reshape(1, -1)

    for i in range(n):
        raw = denorm(sample_imgs[i])
        lbl = sample_lbls[i].item()
        axes[i,0].imshow(raw); axes[i,0].set_title(f'Original\n{CFG.CLASS_NAMES[lbl]}'); axes[i,0].axis('off')
        for c in range(CFG.NUM_CLASSES):
            sv = np.abs(np.transpose(svs[c][i], (1,2,0))).mean(-1)
            sv = (sv - sv.min()) / (sv.max() - sv.min() + 1e-8)
            axes[i,c+1].imshow(raw); axes[i,c+1].imshow(sv, cmap='RdBu_r', alpha=0.6)
            axes[i,c+1].set_title(f'SHAP — {CFG.CLASS_NAMES[c]}'); axes[i,c+1].axis('off')

    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}shap_explanations.png', dpi=150, bbox_inches='tight')
    plt.show(); print("✅ SHAP saved!")

try:
    vis_shap()
except Exception as e:
    print(f"⚠️  SHAP failed: {e}\n   (SHAP on hybrid CNN+ViT models is fragile across versions — "
          f"GradCAM/IG/LIME above already give you solid explainability coverage for the paper.)")

free_mem()

## Cell 13 — Occlusion Sensitivity + XAI Side-by-Side Panel

In [ ]:
def vis_occlusion(n=4):
    n = min(n, len(sample_imgs))
    occ = Occlusion(wrapped)
    fig, axes = plt.subplots(3, n, figsize=(n*4, 12))
    fig.suptitle('Occlusion Sensitivity — Br35H', fontsize=13, fontweight='bold')
    if n == 1:
        axes = axes.reshape(3, 1)
    for i in range(n):
        it  = sample_imgs[i:i+1].to(DEVICE)
        lbl = sample_lbls[i].item()
        att = occ.attribute(it, target=lbl, strides=(3,12,12),
                             sliding_window_shapes=(3,30,30), baselines=0)
        am  = np.abs(att[0].permute(1,2,0).cpu().detach().numpy()).mean(-1)
        am  = (am - am.min()) / (am.max() - am.min() + 1e-8)
        raw = denorm(sample_imgs[i])
        axes[0,i].imshow(raw);            axes[0,i].set_title(CFG.CLASS_NAMES[lbl]); axes[0,i].axis('off')
        axes[1,i].imshow(am, cmap='hot'); axes[1,i].set_title('Occlusion Map');        axes[1,i].axis('off')
        axes[2,i].imshow(raw); axes[2,i].imshow(am, cmap='jet', alpha=0.5)
        axes[2,i].set_title('Overlay'); axes[2,i].axis('off')
    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}occlusion_sensitivity.png', dpi=150, bbox_inches='tight')
    plt.show(); print("✅ Occlusion saved!")

try:
    vis_occlusion()
except Exception as e:
    print(f"⚠️  Occlusion sensitivity failed: {e}")

def xai_panel(idx=0):
    img_t = sample_imgs[idx:idx+1]; lbl = sample_lbls[idx].item()
    raw = denorm(sample_imgs[idx])
    panels = {'Original': raw}

    with GradCAMPlusPlus(model=wrapped, target_layers=[model.cnn.last_conv_layer()]) as cam:
        gc = cam(img_t.to(DEVICE), [ClassifierOutputTarget(lbl)])[0]
    panels['GradCAM++'] = show_cam_on_image(raw, gc, use_rgb=True) / 255.0

    mask = attn_rollout(img_t)
    msk = np.array(Image.fromarray(mask).resize((CFG.IMG_SIZE, CFG.IMG_SIZE), Image.LANCZOS))
    panels['Attn Rollout'] = np.clip(raw*0.5 + plt.cm.jet(msk)[:,:,:3]*0.5, 0, 1)

    it = img_t.to(DEVICE).requires_grad_(True)
    # Move nt_type, nt_samples, stdevs to the attribute method
    nt = NoiseTunnel(IntegratedGradients(wrapped))
    at = nt.attribute(inputs=it, baselines=torch.zeros_like(it), target=lbl, nt_type='smoothgrad_sq', nt_samples=1, stdevs=0.02)
    am = np.abs(at[0].permute(1,2,0).cpu().detach().numpy()).mean(-1)
    am = (am - am.min()) / (am.max() - am.min() + 1e-8)
    panels['Integ. Grad.'] = np.clip(raw*0.5 + plt.cm.hot(am)[:,:,:3]*0.5, 0, 1)

    oc = Occlusion(wrapped).attribute(img_t.to(DEVICE), target=lbl,
                                       strides=(3,12,12), sliding_window_shapes=(3,30,30), baselines=0)
    om = np.abs(oc[0].permute(1,2,0).cpu().detach().numpy()).mean(-1)
    om = (om - om.min()) / (om.max() - om.min() + 1e-8)
    panels['Occlusion'] = np.clip(raw*0.5 + plt.cm.jet(om)[:,:,:3]*0.5, 0, 1)

    fig, axes = plt.subplots(1, len(panels), figsize=(len(panels)*4, 4.5))
    fig.suptitle(f'XAI Comparison — Class: {CFG.CLASS_NAMES[lbl]}', fontsize=13, fontweight='bold')
    for ax, (name, img) in zip(axes, panels.items()):
        ax.imshow(np.clip(img, 0, 1)); ax.set_title(name, fontweight='bold'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}xai_comparison_panel.png', dpi=200, bbox_inches='tight')
    plt.show(); print("✅ XAI comparison panel saved!")

try:
    xai_panel(idx=0)
except Exception as e:
    print(f"⚠️  XAI panel failed: {e}")

free_mem()

## Cell 14 — K-Fold Cross Validation (optional, heavy)

Skipped by default. Set `Config.RUN_HEAVY_CELLS = True` in Cell 4 to run it.

In [ ]:
if not CFG.RUN_HEAVY_CELLS:
    print("⏭️  Skipping K-Fold CV — set Config.RUN_HEAVY_CELLS = True in Cell 4 to run it.")
    print("   This retrains the full hybrid model 5 times and can take hours on a free Colab GPU.")
else:
    # ── Reviewer #1 (CRITICAL): the CV must run ONLY on train_val, never on
    #    the full `df` — `df` still contains the held-out `test_df` rows, so
    #    splitting/indexing against `df` here would leak test images into
    #    CV training/validation folds and invalidate both the CV numbers and
    #    the final test-set numbers reported elsewhere in this notebook.
    #    `train_val` was already reset_index(drop=True) in Cell 6, so the
    #    positional indices kf.split() returns line up with train_val.iloc[].
    K = 5
    tv = train_val  # explicit alias so the leakage fix below is unmissable

    if _HAS_SGKF:
        # Reviewer #5: fold on patient/subject GROUPS, not raw rows, so no
        # inferred group straddles a train/val fold boundary either.
        kf = StratifiedGroupKFold(K, shuffle=True, random_state=CFG.SEED)
        split_iter = kf.split(tv['path'], tv['label_enc'], groups=tv['group_id'])
    else:
        from sklearn.model_selection import GroupKFold
        print("⚠️  sklearn too old for StratifiedGroupKFold — falling back to plain GroupKFold "
              "(group-safe, but folds won't be class-stratified).")
        kf = GroupKFold(K)
        split_iter = kf.split(train_val['path'], train_val['label_enc'], groups=tv['group_id'])

    fold_results = []

    for fold, (tr_idx, va_idx) in enumerate(split_iter, 1):
        print(f"\n── Fold {fold}/{K} ──────────────────────────────────────────")
        # FIX: index into `tv` (== train_val), NOT the full `df` — this is
        # the line that previously caused the test-set leak.
        fd_tr = tv.iloc[tr_idx].reset_index(drop=True)
        fd_va = tv.iloc[va_idx].reset_index(drop=True)
        assert set(fd_tr['group_id']) & set(fd_va['group_id']) == set(), \
            f"Fold {fold}: group leakage between fold-train and fold-val!"

        fd_cw = compute_class_weights(fd_tr).to(DEVICE)
        fd_tr_ldr = DataLoader(MedDataset(fd_tr, train_tfm), CFG.BATCH_SIZE,
                                shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
        fd_va_ldr = DataLoader(MedDataset(fd_va, val_tfm), CFG.BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        fm = HybridModel().to(DEVICE)
        fo = build_optimizer(fm)
        fsch = build_scheduler(fo, CFG.KFOLD_EPOCHS, max(1, CFG.KFOLD_EPOCHS // 5))
        fc = CombLoss(CFG.NUM_CLASSES, fd_cw)
        fsc = make_scaler()
        best_fa = 0.0

        for ep in range(1, CFG.KFOLD_EPOCHS + 1):
            fm.train()
            for imgs, lbls in fd_tr_ldr:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                fo.zero_grad(set_to_none=True)
                with autocast_ctx():
                    ml, cl, vl = fm(imgs)
                    loss = fc(ml, cl, vl, lbls)
                fsc.scale(loss).backward(); fsc.step(fo); fsc.update()
            fsch.step()
            _, va_acc, va_f1, va_auc, *_ = validate(fm, fd_va_ldr, fc)
            if va_auc > best_fa:
                best_fa = va_auc
                torch.save(fm.state_dict(), f'{CFG.CHECKPOINT}fold{fold}_best.pth')
            if ep % 5 == 0 or ep == CFG.KFOLD_EPOCHS:
                print(f"   Ep {ep:02d}: AUC {va_auc:.4f} | Acc {va_acc:.4f}")

        fm.load_state_dict(torch.load(f'{CFG.CHECKPOINT}fold{fold}_best.pth', map_location=DEVICE))
        _, va_acc, va_f1, va_auc, yt, yp, _ = validate(fm, fd_va_ldr, fc)
        fold_results.append({'Fold': fold, 'AUC': va_auc, 'Accuracy': va_acc, 'F1': va_f1,
                              'MCC': matthews_corrcoef(yt, yp), 'Kappa': cohen_kappa_score(yt, yp)})
        print(f"   Best AUC={va_auc:.4f}  Acc={va_acc:.4f}  F1={va_f1:.4f}")
        del fm; free_mem()

    fold_df = pd.DataFrame(fold_results)
    print(f"\n{'='*52}\n  {K}-FOLD CROSS-VALIDATION SUMMARY (train_val only — test set untouched)\n{'='*52}")
    print(fold_df.to_string(index=False))
    for col in ['AUC','Accuracy','F1','MCC','Kappa']:
        print(f"  {col:<12}: {fold_df[col].mean():.4f} ± {fold_df[col].std():.4f}")
    fold_df.to_csv(f'{CFG.RESULTS}kfold_results.csv', index=False)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'{K}-Fold CV Results — Br35H (train_val only)', fontsize=13, fontweight='bold')
    cols_kf = ['#3498db','#e74c3c','#2ecc71']
    folds_labels = [f'Fold {i}' for i in range(1, K+1)]
    for ax, metric, color in zip(axes, ['AUC','Accuracy','F1'], cols_kf):
        vals = fold_df[metric].values
        bars = ax.bar(folds_labels, vals, color=color, alpha=0.85, edgecolor='white')
        ax.axhline(vals.mean(), color='black', ls='--', lw=1.5, label=f'Mean={vals.mean():.3f}')
        for b, v in zip(bars, vals):
            ax.text(b.get_x()+b.get_width()/2, v+0.003, f'{v:.3f}', ha='center', fontsize=9)
        ax.set_title(metric, fontweight='bold'); ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}kfold_visualization.png', dpi=150, bbox_inches='tight')
    plt.show(); print("✅ K-Fold complete!")

## Cell 15 — Ablation Study (optional, heavy) + Feature Visualization (t-SNE / UMAP)

In [ ]:
import numpy as np
# ── Ablation building blocks ─────────────────────
# CNNBranch (Cell 7) always applies CBAM internally, so to isolate CBAM's
# own contribution we need a CBAM-free CNN branch as the true baseline.
class CNNBranchNoCBAM(nn.Module):
    def __init__(self, name=CFG.CNN_BACKBONE, out=CFG.FUSION_DIM):
        super().__init__()
        self.backbone = timm.create_model(name, pretrained=True, num_classes=0, global_pool='')
        with torch.no_grad():
            ch = self.backbone.forward_features(torch.zeros(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)).shape[1]
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Flatten(), nn.LayerNorm(ch),
                                   nn.Linear(ch, out), nn.GELU(), nn.Dropout(CFG.DROPOUT))
    def forward(self, x):
        f = self.backbone.forward_features(x)
        return self.proj(self.pool(f))

class CNNOnly(nn.Module):
    """1. CNN (Baseline) — no CBAM, no ViT, no fusion."""
    def __init__(self):
        super().__init__()
        self.cnn = CNNBranchNoCBAM()
        self.cls = nn.Sequential(nn.LayerNorm(CFG.FUSION_DIM), nn.Dropout(CFG.DROPOUT),
                                  nn.Linear(CFG.FUSION_DIM, CFG.NUM_CLASSES))
    def forward(self, x): return self.cls(self.cnn(x)), None, None

class CNNCBAMOnly(nn.Module):
    """2. CNN + CBAM — isolates CBAM's own effect over the plain CNN baseline."""
    def __init__(self):
        super().__init__()
        self.cnn = CNNBranch()  # CBAM is built into CNNBranch (Cell 7)
        self.cls = nn.Sequential(nn.LayerNorm(CFG.FUSION_DIM), nn.Dropout(CFG.DROPOUT),
                                  nn.Linear(CFG.FUSION_DIM, CFG.NUM_CLASSES))
    def forward(self, x): return self.cls(self.cnn(x)), None, None

class ViTOnly(nn.Module):
    """3. ViT (Baseline)."""
    def __init__(self):
        super().__init__()
        self.vit = ViTBranch()
        self.cls = nn.Sequential(nn.LayerNorm(CFG.FUSION_DIM), nn.Dropout(CFG.DROPOUT),
                                  nn.Linear(CFG.FUSION_DIM, CFG.NUM_CLASSES))
    def forward(self, x): return self.cls(self.vit(x)), None, None

class ConcatFusion(nn.Module):
    """4. CNN + ViT (Concat) — no CBAM, no cross-attention, no gating:
    isolates the value of *any* fusion at all over single-branch models."""
    def __init__(self):
        super().__init__()
        self.cnn = CNNBranchNoCBAM(); self.vit = ViTBranch()
        self.cls = nn.Sequential(nn.LayerNorm(CFG.FUSION_DIM*2),
                                  nn.Linear(CFG.FUSION_DIM*2, CFG.FUSION_DIM),
                                  nn.GELU(), nn.Dropout(CFG.DROPOUT),
                                  nn.Linear(CFG.FUSION_DIM, CFG.NUM_CLASSES))
    def forward(self, x): return self.cls(torch.cat([self.cnn(x), self.vit(x)], -1)), None, None

class CrossAttnOnly(nn.Module):
    """5. CNN + ViT + Cross-Attention — no CBAM, no gating."""
    def __init__(self):
        super().__init__()
        self.cnn = CNNBranchNoCBAM(); self.vit = ViTBranch()
        self.fusion = CrossAttnFusion(CFG.FUSION_DIM)
        self.cls = nn.Sequential(nn.LayerNorm(CFG.FUSION_DIM), nn.Dropout(CFG.DROPOUT),
                                  nn.Linear(CFG.FUSION_DIM, CFG.NUM_CLASSES))
    def forward(self, x):
        cf, vf = self.cnn(x), self.vit(x)
        fused, _ = self.fusion(cf, vf)
        return self.cls(fused), None, None

class GatingOnly(nn.Module):
    """6. CNN + ViT + Learnable Gating — no CBAM, no cross-attention."""
    def __init__(self):
        super().__init__()
        self.cnn = CNNBranchNoCBAM(); self.vit = ViTBranch()
        self.gate = nn.Sequential(nn.Linear(CFG.FUSION_DIM * 2, 2), nn.Softmax(dim=-1))
        self.cls = nn.Sequential(nn.LayerNorm(CFG.FUSION_DIM), nn.Dropout(CFG.DROPOUT),
                                  nn.Linear(CFG.FUSION_DIM, CFG.NUM_CLASSES))
    def forward(self, x):
        cf, vf = self.cnn(x), self.vit(x)
        g = self.gate(torch.cat([cf, vf], dim=-1))
        z = g[:, 0:1] * cf + g[:, 1:2] * vf
        return self.cls(z), None, None

class CrossAttnGating(nn.Module):
    """7. CNN + ViT + Cross-Attention + Gating — everything except CBAM.
    Comparing this row to the full model isolates CBAM's marginal effect
    on top of the complete fusion strategy; comparing it to rows 5 & 6
    isolates the fusion strategy itself."""
    def __init__(self):
        super().__init__()
        self.cnn = CNNBranchNoCBAM(); self.vit = ViTBranch()
        self.fusion = CrossAttnFusion(CFG.FUSION_DIM)
        self.gate = nn.Sequential(nn.Linear(CFG.FUSION_DIM * 2, 2), nn.Softmax(dim=-1))
        self.cls = nn.Sequential(nn.LayerNorm(CFG.FUSION_DIM), nn.Dropout(CFG.DROPOUT),
                                  nn.Linear(CFG.FUSION_DIM, CFG.NUM_CLASSES))
    def forward(self, x):
        cf, vf = self.cnn(x), self.vit(x)
        fused, _ = self.fusion(cf, vf)
        g = self.gate(torch.cat([cf, vf], dim=-1))
        z = g[:, 0:1] * cf + g[:, 1:2] * vf + fused
        return self.cls(z), None, None


def quick_train(ModelCls, name, epochs):
    """Trains one ablation variant under the SAME protocol as the full model:
    train -> checkpoint-select on val AUC -> freeze -> evaluate once on test.

    Reviewer #4 bugfix: previously this function reported `best_auc_ab` from
    whichever epoch had the highest val AUC, but `v_acc`/`v_f1` from
    whatever epoch the training loop happened to end on — two different
    checkpoints. Now Accuracy/F1 are captured in the SAME `if` branch as the
    best AUC, so all three numbers always come from one consistent epoch,
    and we additionally reload that exact checkpoint's weights before the
    final test-set evaluation (rather than reusing in-memory weights from
    the LAST epoch, which may not be the best one)."""
    print(f"\n Ablation: {name}")
    m = ModelCls().to(DEVICE)
    op = optim.AdamW(m.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    cr = CombLoss(CFG.NUM_CLASSES, class_weights)
    sch = build_scheduler(op, CFG.NUM_EPOCHS, CFG.WARMUP_EPOCHS)
    sc = make_scaler()

    best_auc_ab, best_acc_ab, best_f1_ab = 0.0, 0.0, 0.0
    best_state = None
    best_epoch_ab = 0
    patience_counter = 0

    for ep in range(1, epochs + 1):
        m.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            op.zero_grad(set_to_none=True)
            with autocast_ctx():
                ml, cl, vl = m(imgs)
                loss = cr(ml, cl, vl, lbls) if cl is not None else F.cross_entropy(ml, lbls, weight=class_weights)
            sc.scale(loss).backward(); sc.step(op); sc.update()
        sch.step()

        _, v_acc, v_f1, v_auc, *_ = validate(m, val_loader, cr)
        if v_auc > best_auc_ab:
            # FIX: all three metrics captured together, from the same epoch.
            best_auc_ab, best_acc_ab, best_f1_ab = v_auc, v_acc, v_f1
            best_epoch_ab = ep
            patience_counter = 0
            best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            patience_counter += 1
        if patience_counter >= CFG.EARLY_STOPPING_PATIENCE:
            print(f"   Early stopping at epoch {ep}; best val AUC={best_auc_ab:.4f} at epoch {best_epoch_ab}")
            break
        if ep % 5 == 0 or ep == epochs:
            print(f"   Ep {ep}: val AUC={v_auc:.4f}  val Acc={v_acc:.4f}")

    if best_state is not None:
        m.load_state_dict(best_state)

    # Reviewer #2: report the FROZEN, best-val-checkpoint model's performance
    # on the SAME held-out test set the full hybrid model is scored on —
    # never the validation set — so every row is directly comparable.
    _, te_acc_ab, te_f1_ab, te_auc_ab, *_ = validate(m, test_loader, cr)

    del m; free_mem()
    print(f"   Best VAL  : AUC={best_auc_ab:.4f} Acc={best_acc_ab:.4f} F1={best_f1_ab:.4f}  (checkpoint selection only)")
    print(f"   Frozen TEST: AUC={te_auc_ab:.4f} Acc={te_acc_ab:.4f} F1={te_f1_ab:.4f}  (reported below)")
    return {'Model': name, 'Val AUC': best_auc_ab, 'Val Accuracy': best_acc_ab, 'Val F1': best_f1_ab,
            'AUC': te_auc_ab, 'Accuracy': te_acc_ab, 'F1': te_f1_ab,
            'Epoch Budget': CFG.NUM_EPOCHS, 'Best Epoch': best_epoch_ab,
            'Checkpoint Metric': CFG.CHECKPOINT_METRIC,
            'Early-Stopping Patience': CFG.EARLY_STOPPING_PATIENCE}


if not CFG.RUN_HEAVY_CELLS:
    print("⏭️  Skipping the ablation study — set Config.RUN_HEAVY_CELLS = True in Cell 4 to run it.")
    print("   It retrains 7 model variants for a component-isolated comparison against the full hybrid model.")
else:
    abl = [
        quick_train(CNNOnly,         f'1. CNN (Baseline, {CFG.CNN_BACKBONE})', CFG.ABLATION_EPOCHS),
        quick_train(CNNCBAMOnly,     '2. CNN + CBAM',                          CFG.ABLATION_EPOCHS),
        quick_train(ViTOnly,         f'3. ViT (Baseline, {CFG.VIT_BACKBONE})', CFG.ABLATION_EPOCHS),
        quick_train(ConcatFusion,    '4. CNN + ViT (Concat)',                  CFG.ABLATION_EPOCHS),
        quick_train(CrossAttnOnly,   '5. CNN + ViT + Cross-Attention',         CFG.ABLATION_EPOCHS),
        quick_train(GatingOnly,      '6. CNN + ViT + Gating',                  CFG.ABLATION_EPOCHS),
        quick_train(CrossAttnGating, '7. CNN + ViT + Cross-Attention + Gating',CFG.ABLATION_EPOCHS),
    ]
    # Full model: same protocol, reusing Cell 8's training run (best-val-AUC
    # checkpoint, `best_model.pth`, was already loaded into `model` and
    # scored on the SAME test_loader in Cell 10 as te_auc/te_acc/te_f1).
    best_ep = int(np.argmax(history['va_auc'])) if history['va_auc'] else 0
    abl.append({
        'Model': '★ 8. Full Hybrid: CNN+CBAM + ViT + Cross-Attn + Gating (Ours)',
        'Val AUC': best_auc,
        'Val Accuracy': history['va_acc'][best_ep] if history['va_acc'] else float('nan'),
        'Val F1': history['va_f1'][best_ep] if history['va_f1'] else float('nan'),
        'AUC': te_auc, 'Accuracy': te_acc, 'F1': te_f1,
        'Epoch Budget': CFG.NUM_EPOCHS, 'Best Epoch': best_ep + 1,
        'Checkpoint Metric': CFG.CHECKPOINT_METRIC,
        'Early-Stopping Patience': CFG.EARLY_STOPPING_PATIENCE,
    })

    abl_df = pd.DataFrame(abl)

    # Final-model rule: retain only the minimum-complexity architecture whose
    # test AUC is practically equivalent to the best observed result. This is
    # an exploratory internal-test summary; final model selection should use
    # repeated validation-seed results before one final frozen test evaluation.
    AUC_EQUIVALENCE_MARGIN = 0.002
    complexity_order = {
        '1. CNN (Baseline, ' + CFG.CNN_BACKBONE + ')': 1,
        '2. CNN + CBAM': 2,
        '3. ViT (Baseline, ' + CFG.VIT_BACKBONE + ')': 3,
        '4. CNN + ViT (Concat)': 4,
        '5. CNN + ViT + Cross-Attention': 5,
        '6. CNN + ViT + Gating': 5,
        '7. CNN + ViT + Cross-Attention + Gating': 6,
        '★ 8. Full Hybrid: CNN+CBAM + ViT + Cross-Attn + Gating (Ours)': 7,
    }
    abl_df['Complexity'] = abl_df['Model'].map(complexity_order)
    assert abl_df['Complexity'].notna().all(), 'Every ablation must have a complexity rank.'
    best_observed_auc = abl_df['AUC'].max()
    eligible_df = abl_df[abl_df['AUC'] >= best_observed_auc - AUC_EQUIVALENCE_MARGIN].copy()
    selected_model = eligible_df.sort_values(
        ['Complexity', 'AUC', 'Accuracy', 'F1'],
        ascending=[True, False, False, False],
    ).iloc[0]
    print('MINIMUM ADEQUATE ARCHITECTURE (exploratory test summary)')
    print(f"Best observed test AUC: {best_observed_auc:.4f}; equivalence margin: {AUC_EQUIVALENCE_MARGIN:.4f}")
    print(eligible_df[['Model', 'Complexity', 'AUC', 'Accuracy', 'F1']].sort_values(['Complexity', 'AUC'], ascending=[True, False]).to_string(index=False))
    print(f"Selected simplest equivalent candidate: {selected_model['Model']}")
    print(f"\n{'='*70}\n  ABLATION STUDY — all rows: same split / same protocol / TEST-set metrics\n{'='*70}")
    print(abl_df.to_string(index=False))
    abl_df.to_csv(f'{CFG.RESULTS}ablation_study.csv', index=False)

    # Reviewer #3 red-flag check: any single-backbone baseline matching or
    # beating the full model on TEST AUC is a shortcut-learning warning
    # sign, not evidence the architecture "isn't needed" — see the
    # shortcut-learning audit cell below before drawing conclusions either way.
    full_auc = abl_df.iloc[-1]['AUC']
    near_or_above = abl_df.iloc[:-1][abl_df.iloc[:-1]['AUC'] >= full_auc - 1e-4]
    if len(near_or_above):
        print(f"\n⚑ NOTE: the following baseline(s) matched or exceeded the full model's "
              f"test AUC ({full_auc:.4f}):")
        print(near_or_above[['Model', 'AUC']].to_string(index=False))
        print("   This does not automatically mean the architecture adds nothing — but an "
              "AUC this high from a single backbone is also a classic shortcut-learning "
              "signature. Check the Shortcut-Learning Audit cell below before reporting "
              "either result as a genuine improvement.")

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    fig.suptitle('Ablation Study — Component Contribution (TEST set) — Br35H', fontsize=13, fontweight='bold')
    n_bars = len(abl_df)
    cols_abl = plt.cm.viridis(np.linspace(0.15, 0.85, n_bars))
    for ax, metric in zip(axes, ['AUC', 'Accuracy']):
        bars = ax.barh(abl_df['Model'], abl_df[metric], color=cols_abl, edgecolor='white', height=0.6)
        bars[-1].set_edgecolor('#c0392b'); bars[-1].set_linewidth(2.5)
        for b, v in zip(bars, abl_df[metric]):
            ax.text(v + 0.003, b.get_y() + b.get_height()/2, f'{v:.4f}', va='center', fontweight='bold', fontsize=8)
        ax.set_title(metric, fontweight='bold'); ax.grid(axis='x', alpha=0.3)
        ax.set_xlim(max(0, abl_df[metric].min() - 0.05), 1.02)
    plt.tight_layout()
    plt.savefig(f'{CFG.RESULTS}ablation_study.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── Feature space visualisation (cheap — always runs) ────────────────
# Reviewer #6: t-SNE/UMAP are QUALITATIVE visualization tools only. Visual
# cluster separation here is not a validation metric and must never be used
# on its own to support a claim of "the model separates classes well" — use
# the test-set AUC/F1/etc. above for that; treat these plots as an
# illustration for a paper figure, not as evidence.
model.eval()
all_feats, all_lbls = [], []
with torch.no_grad():
    for imgs, lbls, *_ in test_loader:
        _, feats, _ = model(imgs.to(DEVICE), return_feats=True)
        all_feats.append(feats.cpu().numpy())
        all_lbls.extend(lbls.numpy())
feats_np = np.vstack(all_feats); lbls_np = np.array(all_lbls)
print(f"✅ Features: {feats_np.shape}")

print("⚙⚔  t-SNE..."); t0 = time.time()
tsne_emb = TSNE(2, perplexity=min(30, len(feats_np)-1), n_iter=1000,
                 random_state=CFG.SEED, n_jobs=-1).fit_transform(feats_np)
print(f"   Done in {time.time()-t0:.1f}s")

print("⚙⚔  UMAP...")
umap_emb = UMAP(n_components=2, random_state=CFG.SEED, n_neighbors=15).fit_transform(feats_np)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Feature Space Visualization — Br35H (qualitative only — not a validation metric)',
             fontsize=13, fontweight='bold')
palette = plt.cm.tab10(np.linspace(0, 0.9, CFG.NUM_CLASSES))
for ax, emb, title in zip(axes, [tsne_emb, umap_emb], ['t-SNE', 'UMAP']):
    for ci, cn in enumerate(CFG.CLASS_NAMES):
        m = lbls_np == ci
        ax.scatter(emb[m,0], emb[m,1], c=[palette[ci]], label=cn, alpha=0.7, s=25, edgecolors='none')
    ax.set_title(title, fontweight='bold'); ax.legend(markerscale=2)
    ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{CFG.RESULTS}tsne_umap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ t-SNE + UMAP saved!")
print("ℹ️  Reminder: these embeddings are illustrative only, not quantitative evidence.")
free_mem()


## Cell 15b — Shortcut-Learning & XAI Faithfulness Audit (Added)

An AUC of ~1.0 from a single backbone (see the ablation study above) is a
classic shortcut-learning red flag before it's a sign of a great model. The
cell below runs the FROZEN final checkpoint (`model`, loaded from
`best_model.pth` in Cell 10) against masked variants of the same test
images. The quantitative XAI evaluation that follows (Cell 15c) checks the
explanations against random baselines and expert tumor outlines instead of
relying on visual agreement alone.

In [ ]:
# ── Shortcut-Learning Audit: background, borders, intensity, cropping ──────
# This is a diagnostic audit on the frozen best-validation checkpoint. It does
# not modify training. The Otsu mask is a coarse heuristic, not clinical skull stripping.
def _border_mask(height, width, fraction=0.10):
    width_px = max(1, int(min(height, width) * fraction))
    mask = np.zeros((height, width), dtype=bool)
    mask[:width_px, :] = mask[-width_px:, :] = True
    mask[:, :width_px] = mask[:, -width_px:] = True
    return mask

def _artifact_diagnostics(rows):
    diagnostics = []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc='Artifact diagnostics'):
        img_u8 = np.array(Image.open(row['path']).convert('RGB'))
        gray = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY).astype(np.float32)
        brain = _brain_mask(img_u8).astype(bool)
        background = ~brain
        border = _border_mask(*gray.shape)
        diagnostics.append({
            'label': int(row['label_enc']),
            'brain_fraction': float(brain.mean()),
            'background_mean_intensity': float(gray[background].mean()) if background.any() else np.nan,
            'background_std_intensity': float(gray[background].std()) if background.any() else np.nan,
            'border_mean_intensity': float(gray[border].mean()),
            'border_std_intensity': float(gray[border].std()),
            'brain_mean_intensity': float(gray[brain].mean()) if brain.any() else np.nan,
            'image_mean_intensity': float(gray.mean()),
            'image_std_intensity': float(gray.std()),
        })
    artifact_df = pd.DataFrame(diagnostics)
    summary = artifact_df.groupby('label').agg(['mean', 'std', 'count']).round(4)
    summary.to_csv(f'{CFG.RESULTS}background_artifact_summary.csv')
    artifact_df.to_csv(f'{CFG.RESULTS}background_artifact_diagnostics.csv', index=False)
    print('NON-ANATOMICAL ARTIFACT DIAGNOSTICS BY CLASS')
    display(summary)
    means = artifact_df.groupby('label').mean(numeric_only=True)
    if set(means.index) >= {0, 1}:
        print('Absolute class-mean differences (larger values warrant investigation):')
        print((means.loc[1] - means.loc[0]).abs().sort_values(ascending=False).round(4).to_string())
    return summary

@torch.no_grad()
def audit_shortcut_learning(n_samples=200, seed=CFG.SEED):
    rng = np.random.default_rng(seed)
    n_samples = min(n_samples, len(test_df))
    rows = test_df.iloc[rng.choice(len(test_df), size=n_samples, replace=False)]
    variants = ['original', 'brain_only', 'background_only', 'skull_masked']
    results = {variant: {'y': [], 'p': []} for variant in variants}
    model.eval()
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc='Shortcut audit'):
        img_u8 = np.array(Image.open(row['path']).convert('RGB'))
        label = int(row['label_enc'])
        for variant in variants:
            tensor = val_tfm(image=_variant(img_u8, variant))['image'].unsqueeze(0).to(DEVICE)
            logits, _, _ = model(tensor)
            probability = F.softmax(logits, dim=1).cpu().numpy()[0]
            results[variant]['y'].append(label)
            results[variant]['p'].append(probability)
    audit_rows = []
    for variant in variants:
        y = np.asarray(results[variant]['y'])
        p = np.asarray(results[variant]['p'])
        auc = roc_auc_score(y, p[:, 1]) if len(np.unique(y)) == 2 else np.nan
        audit_rows.append({'Variant': variant, 'Accuracy': (p.argmax(1) == y).mean(), 'AUC': auc, 'Samples': len(y)})
    audit_df = pd.DataFrame(audit_rows)
    audit_df.to_csv(f'{CFG.RESULTS}shortcut_audit.csv', index=False)
    print(f"\n{'='*62}\nSHORTCUT-LEARNING AUDIT: {n_samples} held-out images\n{'='*62}")
    print(audit_df.to_string(index=False))
    background_auc = audit_df.loc[audit_df['Variant'] == 'background_only', 'AUC'].iloc[0]
    if background_auc >= 0.65:
        print(f'\nWARNING: background-only AUC={background_auc:.4f} is materially above chance (0.50).')
        print('Possible class-correlated cues include border/padding geometry, cropping or brain-size differences,')
        print('scanner vignetting, compression, and global/background intensity statistics. This is a warning')
        print('signal—not proof of a specific causal artifact—because the mask is a coarse heuristic.')
    else:
        print(f'\nBackground-only AUC={background_auc:.4f}, close to chance.')
    artifact_summary = _artifact_diagnostics(rows)
    return audit_df, artifact_summary

if not CFG.RUN_HEAVY_CELLS:
    print('Skipping shortcut-learning audit — set Config.RUN_HEAVY_CELLS = True to run it.')
else:
    audit_df, artifact_summary = audit_shortcut_learning()

## Cell 15c — Final quantitative XAI evaluation (replaces the earlier faithfulness audit)

This section replaces the earlier deletion/insertion and pointing-game cells. Those cells pooled tumor and no-tumor images, had no random-attribution baseline, and used an Otsu brain mask as the pointing-game reference. The final protocol:

- Seven methods, each reported separately: Grad-CAM++ and Score-CAM on the CBAM-refined map, ViT attention rollout, Integrated Gradients, GradientSHAP, occlusion, and LIME.
- Insertion/deletion AUC (50 steps, Gaussian-blur baseline 31 × 31, σ = 10; black fill as a sensitivity run), compared with a uniform random map and a random map restricted to the brain.
- Results stratified by class (tumor / no tumor), correctness, and whether the image's scan also occurs in train/val.
- Pointing game against the expert tumor outlines distributed with Br35H (801 images, matched by MD5; rotated ellipses handled), with a per-image chance level and energy-in-outline.
- One-sided Wilcoxon tests on scan-cluster means with Holm correction, plus cluster-bootstrap 95% CIs.

All Table 8 and Figure 8 values must come from the files written here (`results/xai_final_blur/`, with `provenance.json`).

In [ ]:
# %% [CELL A] — load the checkpoint and the Br35H expert tumor annotations
import os, sys, json, time
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
import cv2
import br35h_data as bd
import xai_faithfulness_eval as xe

ckpt = torch.load(f'{CFG.CHECKPOINT}best_model.pth', map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['state']); model.eval()
wrapped = wrapped.eval()
print(f"Checkpoint epoch {ckpt.get('epoch')} (val AUC {ckpt.get('auc', float('nan')):.4f}) loaded")

ANN_ROOT = os.path.join(RAW_DATA_DIR, 'Br35H-Mask-RCNN')
if not os.path.isdir(ANN_ROOT):
    # Fallback: fetch only the annotation folder from the dataset author's GitHub repository
    ANN_ROOT = '/content/br35h_ann/Br35H-Mask-RCNN'
    if not os.path.isdir(ANN_ROOT):
        !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/AhmedHamada0/Brain-Tumor-Detection-Dataset /content/br35h_ann
        !cd /content/br35h_ann && git sparse-checkout set Br35H-Mask-RCNN
MASKS = bd.load_br35h_masks(ANN_ROOT, size=CFG.IMG_SIZE)
fr = np.array([v['area_frac'] for v in MASKS.values()])
print(f"Expert masks: {len(MASKS)} images (expected 801); tumor area = "
      f"{100*fr.mean():.1f}% of the image on average (median {100*np.median(fr):.1f}%)")
assert len(MASKS) == 801, "annotation folder incomplete — check ANN_ROOT"


In [ ]:
# %% [CELL B] — build the evaluation items (prediction, mask, brain mask, scan cluster, leakage)
t0 = time.time()
test_paths = test_df['path'].tolist()
trainval_paths = train_df['path'].tolist() + val_df['path'].tolist()
test_clusters, test_leak = bd.leakage_flags(test_paths, trainval_paths)
print(f"Scan clustering done in {time.time()-t0:.0f}s: "
      f"{test_leak.sum()}/{len(test_leak)} test images share a scan with train/val; "
      f"{len(set(test_clusters))} distinct scan clusters in the test set")

test_df_x, aligned_masks = bd.attach_masks(test_df, MASKS)
keep = np.arange(len(test_df_x))
if CFG.XAI_MAX_IMAGES is not None:            # budget run: all annotated tumor images + seeded no-tumor subset
    rng_sel = np.random.default_rng(CFG.SEED)
    pos = np.where(test_df_x['has_mask'].values)[0]
    neg = np.where(test_df_x['label_enc'].values == 0)[0]
    n_pos = min(len(pos), CFG.XAI_MAX_IMAGES // 2)
    keep = np.sort(np.r_[rng_sel.choice(pos, n_pos, replace=False),
                         rng_sel.choice(neg, min(len(neg), CFG.XAI_MAX_IMAGES - n_pos), replace=False)])
    print(f"XAI budget run: {len(keep)} of {len(test_df_x)} test images")
items = []
with torch.no_grad():
    for i, row in test_df_x.reset_index(drop=True).iloc[keep].iterrows():
        img = np.array(Image.open(row['path']).convert('RGB'))
        x = val_tfm(image=img)['image'].unsqueeze(0).to(DEVICE)
        p = F.softmax(wrapped(x), 1)[0, 1].item()
        raw224 = cv2.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE), interpolation=cv2.INTER_LINEAR)  # = A.Resize
        items.append(dict(x=x, label=int(row['label_enc']), pred=int(p > 0.5), prob=p,
                          id=os.path.basename(row['path']), cluster=f"c{test_clusters[i]}",
                          leak=bool(test_leak[i]), brain=bd.brain_mask(raw224),
                          mask=aligned_masks[i] if int(row['label_enc']) == 1 else None))
n_mask = sum(it['mask'] is not None for it in items)
print(f"{len(items)} test items; tumor images with an expert mask: {n_mask} "
      f"({len({it['cluster'] for it in items if it['mask'] is not None})} scan clusters)")
acc = np.mean([it['pred'] == it['label'] for it in items]); print(f"sanity: test accuracy {acc:.4f}")


In [ ]:
# %% [CELL C] — run the evaluation (primary: blur baseline). About 10-15 s per image on a T4.
EXPL = xe.build_explainers(model, wrapped, lime_samples=500)
cfg = dict(xe.CFG, baseline='blur', n_steps=50, n_random_seeds=3, target='pred')
OUT = f'{CFG.RESULTS}xai_final_blur/'
xai_df, xai_summary = xe.evaluate(wrapped, items, EXPL, cfg=cfg, out_dir=OUT)
from IPython.display import Markdown, display
display(Markdown(open(OUT + 'table8.md').read()))

# Sensitivity analysis (report in the supplement): black fill instead of blur.
if CFG.XAI_RUN_BLACK_SENSITIVITY:
    cfg_b = dict(cfg, baseline='black')
    _, xai_summary_black = xe.evaluate(wrapped, items, EXPL, cfg=cfg_b, out_dir=f'{CFG.RESULTS}xai_final_black/')
    print(open(f'{CFG.RESULTS}xai_final_black/table8.md').read())


### Figure 8 — tumor-positive and negative cases, correct and incorrect (fixed selection rule)

In [ ]:
# %% [CELL D] — Figure 8: tumor-positive AND negative cases, correct AND wrong
def pick(kind, k, prefer_mask=True, prefer_clean=True, seed=42):
    rng = np.random.default_rng(seed)
    t, p = {'TP': (1, 1), 'FN': (1, 0), 'TN': (0, 0), 'FP': (0, 1)}[kind]
    idx = [i for i, it in enumerate(items) if it['label'] == t and it['pred'] == p]
    key = lambda i: ((items[i]['mask'] is None) if prefer_mask and t == 1 else 0,
                     items[i]['leak'] if prefer_clean else 0, rng.random())
    chosen, seen = [], set()
    for i in sorted(idx, key=key):            # one image per scan cluster
        if items[i]['cluster'] not in seen:
            chosen.append(i); seen.add(items[i]['cluster'])
        if len(chosen) == k:
            break
    return chosen

cases = {'TP': pick('TP', 3), 'FN': pick('FN', 1), 'TN': pick('TN', 2), 'FP': pick('FP', 1)}
print({k: [items[i]['id'] for i in v] for k, v in cases.items()})
fig8 = xe.figure8(items, EXPL, cases, f'{CFG.RESULTS}figure8_xai_final.png',
                  class_names=('No tumor', 'Tumor'), cfg=cfg)
from IPython.display import Image as _I, display; display(_I(fig8))


In [ ]:
# %% [CELL E] — numbers for the manuscript (Section IV.H) and stale-output guard
S = xai_summary; PG = S['Pointing game (tumor images with expert mask)']
for m in list(EXPL) + [xe.RANDOM_B]:
    t, a, pg = S['Tumor'][m], S['All'][m], PG[m]
    print(f"{m:26s} tumor Ins {t['ins_mean']:.3f} Del {t['dele_mean']:.3f} | gap-random(all) "
          f"{a['gap_minus_random_brain']:+.3f} | pointing {100*pg['hit_rate']:.1f}% "
          f"(chance {100*pg['chance']:.1f}%, n={pg['n_images']} imgs / {pg['n_clusters']} scans)")
print("\nClean-subset check (test images with NO copy of the same scan in train/val):")
for m in list(EXPL) + [xe.RANDOM_B]:
    c = S['No train copy'].get(m)
    if c: print(f"  {m:26s} n={c['n_images']:3d} ({c['n_clusters']} scans)  Ins {c['ins_mean']:.3f}  Del {c['dele_mean']:.3f}")
json.dump({'checkpoint_epoch': ckpt.get('epoch'), 'n_items': len(items), 'n_masked': n_mask,
           'generated': time.strftime('%Y-%m-%d %H:%M')}, open(OUT + 'provenance.json', 'w'), indent=2)
print(f"\nAll Table 8 / Figure 8 numbers must come from {OUT} (see provenance.json).")


## Cell 16 — Test-Time Augmentation + Bootstrap CI + Final Report + ONNX Export

In [ ]:
# Reviewer #6: TTA must be reproducible. The original transforms used
# `A.Rotate(limit=15, p=1)` and `A.RandomBrightnessContrast(p=1)`, both of
# which sample a NEW random value every call — so re-running "the same" TTA
# gave different numbers each time. Fixed here to exact, named transforms:
# a horizontal flip, a fixed +15° rotation, a fixed -15° rotation, and a
# fixed (not randomly sampled) brightness/contrast bump — achieved by
# passing a zero-width (min==max) range to Albumentations, which still
# samples "uniformly" but between two equal bounds, so it always returns
# that exact value.
tta_tfms = {
    'original':  val_tfm,
    'hflip':     A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), A.HorizontalFlip(p=1),
                             A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()]),
    'rot+15':    A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), A.Affine(rotate=(15, 15), p=1),
                             A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()]),
    'rot-15':    A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), A.Affine(rotate=(-15, -15), p=1),
                             A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()]),
    'brighter':  A.Compose([A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
                             A.RandomBrightnessContrast(brightness_limit=(0.15, 0.15),
                                                         contrast_limit=(0.0, 0.0), p=1),
                             A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()]),
}

def tta_predict_single(img_np):
    model.eval()
    ps = []
    with torch.no_grad():
        for name, tfm in tta_tfms.items():
            t = tfm(image=img_np)['image'].unsqueeze(0).to(DEVICE)
            logits, _, _ = model(t)
            ps.append(F.softmax(logits, 1).cpu().numpy())
    return np.mean(ps, 0)[0]

print(f"⚙️  TTA inference on test set (deterministic transforms: {list(tta_tfms.keys())})...")
tta_probs, tta_labels = [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img = np.array(Image.open(row['path']).convert('RGB'))
    tta_probs.append(tta_predict_single(img))
    tta_labels.append(int(row['label_enc']))

tta_probs = np.array(tta_probs); tta_labels = np.array(tta_labels)
tta_preds = tta_probs.argmax(1)
tta_acc = (tta_preds == tta_labels).mean()
tta_auc = roc_auc_score(tta_labels, tta_probs[:,1])
tta_f1 = f1_score(tta_labels, tta_preds, average='weighted')
print(f"\n📊 TTA Results (reported separately from the single-pass test numbers below):")
print(f"   Single-pass Accuracy : {te_acc:.4f}")
print(f"   Single-pass AUC-ROC  : {te_auc:.4f}")
print(f"   Single-pass F1-Score : {te_f1:.4f}")
print(f"   TTA Accuracy         : {tta_acc:.4f}  (Δ{tta_acc-te_acc:+.4f})")
print(f"   TTA AUC-ROC          : {tta_auc:.4f}  (Δ{tta_auc-te_auc:+.4f})")
print(f"   TTA F1-Score         : {tta_f1:.4f}  (Δ{tta_f1-te_f1:+.4f})")

def bootstrap_auc_ci(yt, ys, n=1000):
    aucs = []; rng = np.random.default_rng(CFG.SEED)
    for _ in range(n):
        idx = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[idx])) < 2:
            continue
        ysc = ys[idx]
        aucs.append(roc_auc_score(yt[idx], ysc[:,1]))
    return np.percentile(aucs, [2.5, 97.5])

ci = bootstrap_auc_ci(y_true, y_prob)
print(f"\n📐 AUC 95% CI (Bootstrap, single-pass): [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"   Report as: AUC = {te_auc:.4f} (95% CI: {ci[0]:.4f}–{ci[1]:.4f})")

report = {
    'Dataset':               'Br35H (binary: tumor / no tumor)',
    'Accuracy':               f'{te_acc:.4f} ({te_acc*100:.2f}%)',
    'AUC-ROC':                f'{te_auc:.4f} (95% CI: {ci[0]:.4f}\u2013{ci[1]:.4f})',
    'F1-Score (Weighted)':    f'{te_f1:.4f}',
    'Balanced Accuracy':      f'{balanced_accuracy_score(y_true, y_pred):.4f}',
    'MCC':                    f'{matthews_corrcoef(y_true, y_pred):.4f}',
    "Cohen's Kappa":          f'{cohen_kappa_score(y_true, y_pred):.4f}',
    'Precision (Weighted)':   f'{precision_score(y_true, y_pred, average="weighted"):.4f}',
    'Recall (Weighted)':      f'{recall_score(y_true, y_pred, average="weighted"):.4f}',
    '--- TTA (deterministic, reported separately) ---': '',
    'TTA Accuracy':           f'{tta_acc:.4f}',
    'TTA AUC-ROC':            f'{tta_auc:.4f}',
    'TTA F1-Score':           f'{tta_f1:.4f}',
    '--- Architecture ---':   '',
    'CNN Backbone':           CFG.CNN_BACKBONE,
    'ViT Backbone':           CFG.VIT_BACKBONE,
    'Fusion Method':          'Cross-Attention + Learnable Gating',
    'Parameters':             f'{tp:,}',
    'Input Size':             f'{CFG.IMG_SIZE}\u00d7{CFG.IMG_SIZE}',
}
if CFG.RUN_HEAVY_CELLS and 'fold_df' in dir():
    report['--- K-Fold CV (train_val only, patient-grouped) ---'] = ''
    report['K-Fold AUC'] = f'{fold_df["AUC"].mean():.4f} \u00b1 {fold_df["AUC"].std():.4f}'
    report['K-Fold Accuracy'] = f'{fold_df["Accuracy"].mean():.4f} \u00b1 {fold_df["Accuracy"].std():.4f}'
if 'audit_df' in dir():
    report['--- Shortcut-Learning Audit ---'] = ''
    for _, r in audit_df.iterrows():
        report[f'Shortcut check: {r["Variant"]}'] = f'Acc={r["Accuracy"]:.4f} AUC={r["AUC"]:.4f}'
if 'xai_summary' in dir():
    report['--- Final XAI evaluation (Cell 15c; tumor images, blur baseline) ---'] = ''
    _pg = xai_summary['Pointing game (tumor images with expert mask)']
    for _m, _t in xai_summary['Tumor'].items():
        _p = _pg.get(_m, {})
        _hit = f", pointing {100*_p['hit_rate']:.1f}% (chance {100*_p['chance']:.1f}%, n={_p['n_images']})" if _p.get('n_images') else ''
        report[f'XAI {_m}'] = f"Ins={_t['ins_mean']:.4f} Del={_t['dele_mean']:.4f}{_hit}"

rdf = pd.DataFrame(report.items(), columns=['Metric', 'Value'])
rdf.to_csv(f'{CFG.RESULTS}final_report.csv', index=False)

print(f"\n{'='*55}")
print(f"  \U0001F4CB FINAL EXPERIMENT REPORT")
print(f"  A Novel Hybrid Deep Learning\u2013Vision Transformer Model")
print(f"  with Explainable AI for Brain Tumor Classification (Br35H)")
print(f"{'='*55}")
for k, v in report.items():
    print(f"  {k:<25}: {v}")
print(f"{'='*55}")

# ── ONNX export — wrapped so a failure here doesn't cost you the report above ──
try:
    dummy = torch.randn(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(DEVICE)
    torch.onnx.export(wrapped, dummy, f'{CFG.RESULTS}hybrid_model.onnx',
                       opset_version=17, input_names=['image'], output_names=['logits'],
                       dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}})
    print(f"\n✅ ONNX model exported!")
except Exception as e:
    print(f"\n⚠️  ONNX export failed (non-critical — your trained .pth checkpoint is still saved): {e}")

print(f"\n📁 All outputs saved to: {CFG.RESULTS}")
print(f"\n📋 Generated files:")
for f in sorted(os.listdir(CFG.RESULTS)):
    sz = os.path.getsize(os.path.join(CFG.RESULTS, f))
    print(f"   {f:<45} ({sz//1024} KB)")
print(f"\n🎉 EXPERIMENT COMPLETE — ready for write-up!")

In [ ]:
print('Installing reportlab...')
!pip install -q reportlab
print('✅ reportlab installed!')

### PDF Summary of Experiment Report

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.pagesizes import letter

def generate_pdf_report(report_data, output_path):
    doc = SimpleDocTemplate(output_path, pagesize=letter)
    styles = getSampleStyleSheet()

    # Custom style for title
    styles.add(ParagraphStyle(name='ReportTitle',
                              fontSize=18,
                              leading=22,
                              alignment=1, # Center alignment
                              spaceAfter=20))

    # Custom style for section headers
    styles.add(ParagraphStyle(name='SectionHeader',
                              fontSize=14,
                              leading=18,
                              spaceAfter=12,
                              spaceBefore=12,
                              fontName='Helvetica-Bold'))

    # Custom style for key-value pairs
    styles.add(ParagraphStyle(name='Metric',
                              fontSize=10,
                              leading=14,
                              fontName='Helvetica'))

    story = []

    # Title
    story.append(Paragraph('Final Experiment Report', styles['ReportTitle']))
    story.append(Paragraph('A Novel Hybrid Deep Learning–Vision Transformer Model', styles['SectionHeader']))
    story.append(Paragraph('with Explainable AI for Brain Tumor Classification (Br35H)', styles['SectionHeader']))
    story.append(Spacer(1, 0.2 * inch))

    # Report content
    for k, v in report_data.items():
        if k.startswith('---'): # Section separator
            story.append(Spacer(1, 0.1 * inch))
            story.append(Paragraph(k.replace('--- ', '').replace(' ---', '').strip(), styles['SectionHeader']))
            story.append(Spacer(1, 0.05 * inch))
        else:
            story.append(Paragraph(f"<b>{k}:</b> {v}", styles['Metric']))

    doc.build(story)
    print(f"✅ PDF report generated: {output_path}")

output_pdf_path = f'{CFG.RESULTS}final_experiment_report_summary.pdf'
generate_pdf_report(report, output_pdf_path)

# Final Publication Validation Protocol (Added)

This section replaces unsupported benchmark/upgrade cells and enforces a publication-grade workflow.

Order of validation:

1. Near-duplicate image checking
2. Clean final training run
3. EMA checkpoint validation
4. Controlled ablation with identical settings
5. Correct XAI target selection
6. Rerun all XAI methods from the final checkpoint
7. Remove stale outputs and placeholder results
8. Multi-seed validation and external validation (if available)

Only results generated after a clean runtime restart should be reported in the manuscript.


In [ ]:

# 2. Final clean run requirement
# Before publication:
# Runtime -> Restart session -> Run all

import random, numpy as np, torch

def set_final_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_final_seed(42)
print("Final reproducible seed set.")


## 4. Controlled Ablation Rules

All ablation models must use:

- identical train/validation/test split
- identical preprocessing
- identical augmentation
- identical optimizer strategy
- identical number of epochs
- identical evaluation set

Do not compare validation metrics for baselines against test metrics for the proposed model.


In [ ]:

# 5. Correct XAI target selection
# Explain what the model predicted, not only the ground truth label.

def get_xai_target(logits):
    predicted_class = int(torch.argmax(logits, dim=1).item())
    return predicted_class

# For misclassified cases report both:
# - explanation for predicted class
# - explanation for true class


## 6. XAI Reproducibility Rule

After the final checkpoint is selected:

1. Reload the final checkpoint.
2. Re-run Grad-CAM++.
3. Re-run Score-CAM.
4. Re-run attention rollout.
5. Re-run SHAP/LIME.
6. Generate all figures from this same checkpoint.

Never combine XAI outputs generated from different training runs.


## 7. Removed Unsupported Results

The following were removed from the notebook:

- dummy benchmark values for ResNet/DenseNet/EfficientNet/ViT/Swin
- unsupported model comparison charts
- placeholder publication upgrade cells
- stale outputs from previous executions

Only experimentally generated results should appear in the paper.


In [ ]:

# 8. Multi-seed evaluation template

SEEDS = [42, 123, 2024, 7, 99]

# For each seed:
# 1. reset weights
# 2. train the model
# 3. evaluate on untouched test set
# 4. store Accuracy, AUC, F1, MCC, Kappa

# Report:
# mean ± standard deviation

# External validation:
# If another MRI dataset is available, evaluate without retraining.



# Publication XAI Figure Protocol

Generate XAI panels for:

1. Correctly classified sample
2. Misclassified sample

Layout:

Original MRI |
Grad-CAM++ |
Score-CAM |
Attention Rollout |
Integrated Gradients

Report:
- predicted class
- confidence score
- explanation consistency


### Legacy XAI panels removed

The earlier "low-faithfulness", "high-faithfulness" and "misclassified sample" panels depended on the old brain-mask pointing game and showed only no-tumor samples. They are replaced by Figure 8 in Cell 15c, which uses a fixed selection rule over tumor-positive and negative, correct and incorrect cases, with the expert tumor outline drawn on every panel.

In [ ]:
# Export experiment tables (uses the DataFrames this notebook actually
# produces — the previous version referenced undefined `ablation_results` /
# `comparison_results` names and would NameError if called).
import pandas as pd

def export_tables():
    if 'abl_df' in dir() and len(abl_df):
        abl_df.to_csv(f'{CFG.RESULTS}ablation_results_export.csv', index=False)
        print(f"   Saved {CFG.RESULTS}ablation_results_export.csv ({len(abl_df)} rows)")
    else:
        print("   ⚠️  abl_df not found — run the ablation study cell first "
              "(requires Config.RUN_HEAVY_CELLS = True).")

    if 'fold_df' in dir() and len(fold_df):
        fold_df.to_csv(f'{CFG.RESULTS}model_comparison_export.csv', index=False)
        print(f"   Saved {CFG.RESULTS}model_comparison_export.csv ({len(fold_df)} rows)")
    else:
        print("   ⚠️  fold_df not found — run the K-Fold CV cell first "
              "(requires Config.RUN_HEAVY_CELLS = True).")

    print("Tables exported.")

# export_tables()

In [ ]:
if 'abl_df' in dir():
    display(abl_df)
else:
    print("⏭️  abl_df not defined — run the ablation study cell "
          "(Config.RUN_HEAVY_CELLS = True) before this cell.")

In [ ]:
if 'fold_df' in dir():
    display(fold_df)
    print(f"\nMean and Standard Deviation for K-Fold Cross-Validation Metrics:\n{'-'*60}")
    for col in ['AUC','Accuracy','F1','MCC','Kappa']:
        print(f"  {col:<12}: {fold_df[col].mean():.4f} \u00b1 {fold_df[col].std():.4f}")
    print("\nYou can find the bar chart visualization of these metrics in the output of the K-Fold CV cell.")
else:
    print("⏭️  fold_df not defined — run the K-Fold CV cell "
          "(Config.RUN_HEAVY_CELLS = True) before this cell.")

### Displaying Ablation and K-Fold Results

In [ ]:
if 'abl_df' in globals():
    print("\nAblation Study Results:")
    display(abl_df)
else:
    print("Ablation DataFrame (abl_df) not found. Please ensure the ablation study cell (Cell 15) has been run with `CFG.RUN_HEAVY_CELLS = True`.")

if 'fold_df' in globals():
    print("\nK-Fold Cross-Validation Results:")
    display(fold_df)
else:
    print("K-Fold DataFrame (fold_df) not found. Please ensure the K-Fold CV cell (Cell 14) has been run with `CFG.RUN_HEAVY_CELLS = True`.")

## Comprehensive Experiment Summary Report

This report consolidates the key findings from the training, evaluation, K-Fold cross-validation, ablation study, shortcut learning audit, and XAI faithfulness analysis of the Hybrid CNN-ViT model for Brain Tumor Classification on the Br35H dataset.

### 1. Overall Model Performance (Test Set)

These metrics represent the model's performance on the unseen test set, providing a robust evaluation of its generalization capabilities. Test-Time Augmentation (TTA) results are also included to show potential improvements from ensemble predictions over augmented views.

In [ ]:
import pandas as pd

# Assuming 'report', 'te_acc', 'te_auc', 'te_f1', 'ci', 'tta_acc', 'tta_auc', 'tta_f1' are available from kernel state

performance_data = {
    'Metric': [
        'Accuracy',
        'AUC-ROC (95% CI)',
        'F1-Score (Weighted)',
        'Balanced Accuracy',
        'MCC',
        'Cohen\'s Kappa',
        'Precision (Weighted)',
        'Recall (Weighted)',
        '--- TTA Enhanced Results ---',
        'TTA Accuracy',
        'TTA AUC-ROC',
        'TTA F1-Score'
    ],
    'Value': [
        report['Accuracy'],
        report['AUC-ROC'],
        report['F1-Score (Weighted)'],
        report['Balanced Accuracy'],
        report['MCC'],
        report['Cohen\'s Kappa'],
        report['Precision (Weighted)'],
        report['Recall (Weighted)'],
        '',
        report['TTA Accuracy'],
        report['TTA AUC-ROC'],
        report['TTA F1-Score']
    ]
}
performance_df = pd.DataFrame(performance_data)
display(performance_df)

### 2. K-Fold Cross-Validation Summary

The 5-Fold Cross-Validation was performed on the `train_val` dataset to assess model stability and robustness across different data partitions. The results are patient-grouped to prevent data leakage at the subject level.

In [ ]:
if 'fold_df' in globals():
    print("\nK-Fold Cross-Validation Metrics (Mean \u00b1 Std Dev):\n")
    kfold_summary = pd.DataFrame({
        'Metric': ['AUC', 'Accuracy', 'F1', 'MCC', 'Kappa'],
        'Mean \u00b1 Std Dev': [
            f"{fold_df['AUC'].mean():.4f} \u00b1 {fold_df['AUC'].std():.4f}",
            f"{fold_df['Accuracy'].mean():.4f} \u00b1 {fold_df['Accuracy'].std():.4f}",
            f"{fold_df['F1'].mean():.4f} \u00b1 {fold_df['F1'].std():.4f}",
            f"{fold_df['MCC'].mean():.4f} \u00b1 {fold_df['MCC'].std():.4f}",
            f"{fold_df['Kappa'].mean():.4f} \u00b1 {fold_df['Kappa'].std():.4f}"
        ]
    })
    display(kfold_summary)
    print("\nIndividual Fold Results:")
    display(fold_df)
else:
    print("K-Fold Cross-Validation results are not available as the `fold_df` DataFrame was not found.")

### 3. Ablation Study Summary

This study evaluates the contribution of different architectural components by training and evaluating various model variants under the same protocol. It helps in understanding the impact of each design choice on the overall performance.

In [ ]:
if 'abl_df' in globals():
    display(abl_df[['Model', 'AUC', 'Accuracy', 'F1', 'Complexity']].sort_values(by='Complexity'))

    if 'selected_model' in globals():
        print(f"\nMinimum Adequate Architecture (exploratory test summary): {selected_model['Model']}")
    else:
        print("\nSelected minimum adequate architecture not available.")
else:
    print("Ablation study results are not available as the `abl_df` DataFrame was not found.")

### 4. Final XAI Evaluation (Table 8)

Per-method insertion/deletion AUC on tumor and no-tumor images, the gap relative to the in-brain random map, Holm-adjusted p-values, and the pointing game against the expert tumor outlines with its chance level. Produced by Cell 15c.

In [ ]:
if 'xai_summary' in globals():
    from IPython.display import Markdown, display
    display(Markdown(open(f'{CFG.RESULTS}xai_final_blur/table8.md').read()))
else:
    print("Final XAI results are not available — run Cell 15c first.")

### 5. Shortcut-Learning Audit

This audit evaluates whether the model relies on non-anatomical features (e.g., background, borders) for its predictions, which could indicate shortcut learning rather than genuine understanding of tumor features. This is assessed by evaluating the model's performance on images where only specific regions (e.g., brain-only, background-only) are retained.

In [ ]:
if 'audit_df' in globals():
    display(audit_df)

    if 'artifact_summary' in globals():
        print("\nNon-Anatomical Artifact Diagnostics by Class (Summary):\n")
        display(artifact_summary)
    else:
        print("Artifact summary not available.")
else:
    print("Shortcut-learning audit results are not available.")

### 6. XAI Score Distributions (Tumor Images)

Per-image insertion and deletion AUCs for each method and for the two random baselines on tumor-positive test images.

In [ ]:
if 'xai_df' in globals() and len(xai_df):
    import matplotlib.pyplot as plt, seaborn as sns
    d = xai_df[xai_df.label == 1]            # tumor images: the stratum that discriminates methods
    order = list(dict.fromkeys(d['method']))
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
    for ax, col, title in zip(axes, ['ins', 'dele'], ['Insertion AUC (higher = better)', 'Deletion AUC (lower = better)']):
        sns.boxplot(data=d, x='method', y=col, order=order, ax=ax, color='lightsteelblue', fliersize=2)
        ax.set_title(f'{title} — tumor images'); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=40)
        for lab in ax.get_xticklabels(): lab.set_ha('right')
    plt.tight_layout(); plt.savefig(f'{CFG.RESULTS}xai_final_distributions.png', dpi=150, bbox_inches='tight'); plt.show()
    display(d.groupby('method')[['ins', 'dele', 'gap']].describe().round(3))
else:
    print("Per-image XAI results are not available — run Cell 15c first.")

### 7. Key Configuration Parameters

Important parameters used during the experiment.

In [ ]:
import pandas as pd

config_params = {
    'Parameter': [
        'CNN Backbone',
        'ViT Backbone',
        'Fusion Method',
        'Total Parameters',
        'Input Image Size',
        'Batch Size',
        'Number of Epochs',
        'Learning Rate',
        'Low-Memory Mode'
    ],
    'Value': [
        CFG.CNN_BACKBONE,
        CFG.VIT_BACKBONE,
        'Cross-Attention + Learnable Gating',
        f'{tp:,}',
        f'{CFG.IMG_SIZE}×{CFG.IMG_SIZE}',
        CFG.BATCH_SIZE,
        CFG.NUM_EPOCHS,
        CFG.LR,
        CFG.LOW_MEMORY_MODE
    ]
}
config_df = pd.DataFrame(config_params)
display(config_df)

This report provides a consolidated overview of the model's performance, architectural contributions, and interpretability metrics. For detailed plots and raw data, please refer to the files saved in the `CFG.RESULTS` directory.

In [ ]:
# ======================================================================================
# Cell 15d  (add AFTER Cell 15; fills the red "TBD" cells of Table 5)
# Per-variant weighted precision, scan-cluster bootstrap AUC CI, and paired significance
# tests against the full model (McNemar exact + paired cluster-bootstrap dAUC, Holm).
# ======================================================================================
# STEP 0 - two one-line edits inside quick_train() in Cell 15, then re-run Cell 15:
#   (a) replace the final evaluation line with
#         _, te_acc_ab, te_f1_ab, te_auc_ab, yt_ab, yp_ab, ypr_ab = validate(m, test_loader, cr)
#         PRED[name] = (yt_ab, yp_ab, ypr_ab)
#   (b) define  PRED = {}  once, before the `abl = [...]` list.
#   For the full model add after Cell 10:  PRED['full'] = (y_true, y_pred, y_prob)
# NOTE: the ablation models were not saved (del m) and cudnn.benchmark is non-deterministic,
# so a re-run can shift the numbers slightly. If it does, refresh ALL of Table 5 from this
# cell's output, not only the TBD cells.
# ======================================================================================
import numpy as np
from scipy.stats import binomtest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

groups = test_df['group_id'].values            # same order as test_loader (shuffle=False)
uniq, inv = np.unique(groups, return_inverse=True)
rng = np.random.default_rng(CFG.SEED)
B = 1000   # cluster bootstrap: resample scan clusters with replacement, keep all images of a cluster

yt_f, yp_f, ypr_f = PRED['full']
names = [k for k in PRED if k != 'full']
rows, pvals = {}, {}
for k in names:
    yt, yp, ypr = PRED[k]
    assert (yt == yt_f).all()
    prec = precision_score(yt, yp, average='weighted')
    rec  = recall_score(yt, yp, average='weighted')       # equals accuracy
    f1   = f1_score(yt, yp, average='weighted')
    auc  = roc_auc_score(yt, ypr[:, 1])
    # McNemar exact test (discordant pairs) vs. the full model
    a = int(((yp == yt) & (yp_f != yt_f)).sum()); b = int(((yp != yt) & (yp_f == yt_f)).sum())
    p_mc = 1.0 if a + b == 0 else binomtest(a, a + b, 0.5).pvalue
    # paired cluster-bootstrap test of the AUC difference
    d = []
    for _ in range(B):
        c = rng.integers(0, len(uniq), len(uniq))
        ix = np.concatenate([np.where(inv == j)[0] for j in c])
        if len(np.unique(yt[ix])) == 2:
            d.append(roc_auc_score(yt[ix], ypr[ix, 1]) - roc_auc_score(yt_f[ix], ypr_f[ix, 1]))
    d = np.array(d); p_auc = min(1.0, 2 * min((d >= 0).mean(), (d <= 0).mean()))
    rows[k] = dict(prec=prec, rec=rec, f1=f1, auc=auc, p_mc=p_mc, p_auc=p_auc)
# Holm correction over the 7 comparisons (separately for the two tests)
def holm(p):
    o = np.argsort(p); adj = np.empty(len(p)); run = 0
    for r, i in enumerate(o):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1, run)
    return adj
ks = list(rows); pm = holm(np.array([rows[k]['p_mc'] for k in ks])); pa = holm(np.array([rows[k]['p_auc'] for k in ks]))
print("variant | weighted precision | recall(=acc) | F1 | p_McNemar(Holm) | p_dAUC(Holm)")
for k, a_, b_ in zip(ks, pm, pa):
    r = rows[k]
    print(f"{k} | {r['prec']:.4f} | {r['rec']:.4f} | {r['f1']:.4f} | {a_:.3f} | {b_:.3f}")
# Paste Precision into column 4 and report e.g. "0.9xx / 0.9xx" (McNemar / dAUC) in the last column.